In [ ]:
# Mount Google Drive -- every path used later in this notebook
# (VIRAT_DATA / VIRAT_Results, UCF_Crime_Subset / UCF_Crime_Results, and
# the UCA caption cache) lives under /content/drive/MyDrive/, so this has
# to run before anything that reads or writes those paths.
from google.colab import drive

drive.mount('/content/drive')


# Fixed version — change log

Every change below is also marked with a `# FIX:` comment at the relevant line.

## v6 — diagnosing and cutting an hour-long single-video run

**Diagnostics (see them on your next run, before anything else):**
- The pipeline cell now prints GPU status (`nvidia-smi`) right after the
  paths. If it says "no GPU detected," that alone explains an hour-long run
  -- YOLO and two 7B models on CPU are each roughly 1-2 orders of magnitude
  slower than on any Colab GPU tier. Check this before anything below.
- `run_pipeline_for_video` now prints a `[timing]` breakdown (seconds and %
  of total) for copy / extract / detect / describe, so you can see directly
  which stage the hour is actually going to instead of guessing.
- `run_phase2` now prints call counts and average latency per VLM/text call.
  A healthy GPU call with a handful of downscaled images typically lands in
  the low single digits of seconds; 20-60s+ per call points at CPU
  inference, not at anything tunable in the prompt or chunking.

**Actual speedups:**
- **VLM images are now downscaled before being sent to the vision model**
  (`VLM_IMAGE_MAX_WIDTH = 960`, in the VLM-pipeline cell). Qwen2.5-VL's
  token cost scales with image resolution; frames straight out of ffmpeg are
  at the video's native resolution (often 1920x1080+), so every chunk was
  paying for far more pixels than an activity description needs.
  Deliberately implemented as a separate copy made only for the VLM call
  (`_load_vlm_image_bytes`), NOT by resizing at extraction time -- resizing
  at extraction would put every predicted bbox in different pixel
  coordinates than VIRAT's native-resolution ground truth, silently
  corrupting every IoU comparison in evaluation. This is the same class of
  bug as the fps mismatch fixed earlier, just for space instead of time, so
  it was kept strictly out of the detection/eval path.
- **Segments shorter than `MIN_SEGMENT_DURATION` (default 0.5s) are dropped
  before Phase 2.** A 1-2 frame tracker blip still costs a full VLM round
  trip if nothing filters it out first. `process_segments` prints how many
  it drops. TRADEOFF, stated explicitly rather than hidden: a genuinely
  brief real activity under this threshold is also skipped and will show up
  as a miss in evaluation -- this is conservative by default; raise or
  lower it after checking the printed drop count against a few inference
  JSONs.

## v5 — further optimization + redundancy cleanup

**Compute:**
- **YOLO model was reloaded from disk for every video.** `detect_people`
  constructed a fresh `YOLO(model_name)` on every call; across an N-video
  dataset run that's N weight loads for weights that never change. It's now
  cached and loaded once, reused for the rest of the run. This required a
  correctness fix alongside it: `persist=True` was changed to `persist=False`,
  because persisting tracker state across separate `.track()` calls only
  matters once the model is *reused* -- with a fresh model every call (the
  old behavior) there was nothing to persist from, so it was silently
  harmless; with a cached model, leaving it `True` would leak track state
  from the end of one video into the start of the next unrelated one.
- **Per-box tensor access replaced with per-frame batched conversion.**
  The original called `.item()` / tensor-indexing separately for every box's
  id, confidence, and 4 bbox coordinates -- each a GPU→CPU sync point on a
  CUDA tensor. Now `xyxy`/`conf`/`id` are each pulled to CPU as one batched
  numpy array per *frame*, so a frame with 10 people costs 3 syncs instead
  of ~60.
- **VLM eviction between passes is now conditional, not automatic.** The
  previous version force-evicted the vision model after every single
  video's vision pass (`keep_alive=0`), then had to pay a full reload for
  the next video's vision pass -- worthwhile on a small-VRAM GPU where two
  7B models don't coreside, pure waste on this notebook's declared Colab
  A100 runtime, which has room for both simultaneously. Gated behind
  `UNLOAD_VLM_BETWEEN_PASSES` (default `False`); flip to `True` if you move
  to a smaller GPU and start seeing OOM/thrash.
- **`append_metrics_row` no longer reads the whole CSV on every call.** It
  parsed the entire existing file with `DictReader` on every single append
  just to compare headers -- O(file size) work per video for a check that's
  almost always a no-op. Now it reads only the first line; the full
  read+rewrite only happens on the rare path where the metrics schema
  actually changed.

**Redundancy / dead code removed:**
- `CLASSES` was defined identically in two cells (the VLM-pipeline cell and
  the evaluation cell) that run in the same kernel. Down to one definition;
  the second cell now relies on it the same way it already relies on
  `extract_frames`, `detect_people`, etc. being defined upstream.
- `OLLAMA_HOST` and an `/api/tags` health-check function were each defined
  twice (once in the Ollama-startup cell, once in the VLM-pipeline cell).
  Down to one definition, reused the same way.
- Removed two unused imports (`math` in the VLM-pipeline cell, `pathlib.Path`
  in the evaluation cell) left over from earlier drafts.

## v4 — pipeline visibility (fixes the "stuck at 0%" hang-vs-slow ambiguity)

Between `Processing videos: 0%` and the first `detections:` print, the
pipeline used to be completely silent while `ffmpeg` read the video straight
off the Drive FUSE mount -- a slow-but-working extraction and a genuine hang
looked identical. `run_pipeline_for_video` now:
- Prints a `[1/4]..[4/4]` stage marker before each major step (copy, extract,
  detect, describe), so silence longer than a few seconds tells you something
  real is wrong, not "still working."
- Copies the video to **local disk first**, with a byte-level `tqdm` progress
  bar, then runs `ffmpeg` against the local copy. Sequential reads off Drive
  via FUSE are the slowest link in this pipeline; local disk is faster and,
  more importantly, now visible instead of a black box.
- Deletes the local video copy after extraction (raw video isn't needed past
  that point), and `clear_pipeline_dirs()` now also sweeps any leftover copy
  from a run that crashed mid-video, so copies don't accumulate on local disk
  across a multi-video run.

## v3 — paths pointed at your actual Drive folders

- `DATA_ROOT = "/content/drive/MyDrive/VIRAT_DATA"`, `VIDEO_DIR = DATA_ROOT` —
  videos are read directly from your existing `VIRAT_DATA` folder, no
  subfolder assumed.
- **Assumption, please verify:** `ANNOTATIONS_DIR = VIRAT_DATA/annotations`.
  Only the videos' location was given; if your VIRAT `.viratdata.events.txt`
  files live somewhere else, change `ANNOTATIONS_DIR` at the top of the
  pipeline cell. If you don't have ground-truth annotations at all and just
  want tracking + descriptions, you can skip the `eval_data(...)` call inside
  `main()` — `run_pipeline_for_video()` alone gives you the JSON descriptions.
- **Results now land in their own folder**, `/content/drive/MyDrive/VIRAT_Results/`,
  sitting next to (not inside) `VIRAT_DATA` so your input data stays untouched:
  - `VIRAT_Results/inference/phase2_output_<video>.json` — one file per video,
    the full per-person tracking + natural-language descriptions.
  - `VIRAT_Results/eval-results.csv` — one row per video's metrics.
  - `VIRAT_Results/eval-results-dataset-summary.csv` — one row, dataset-wide totals.
- The pipeline now **prints all resolved paths** when the cell runs, so it's
  visible at a glance where it's reading from and writing to.

## v2 — path reorganization + reference-photo matching removed

**Paths.** Three roots now, instead of scattered ad-hoc strings:
- `DATA_ROOT` (Drive) — read-only source data: VIRAT videos + annotations
  (also fixed a stray `annotations (1)` directory name to `annotations`).
- `PROJECT_ROOT` / `INFERENCE_DIR` (Drive) — the only things written to
  Drive: per-video inference JSON and the two result CSVs.
- `WORK_DIR = "/content/work"` (**local Colab disk, not Drive**) — frames,
  boxed frames, and person crops. These are thousands of small JPEGs per
  video, fully ephemeral, and rewritten per video by `clear_pipeline_dirs()`.
  Writing that volume of small files to a Drive FUSE mount is the single
  biggest avoidable slowdown in this pipeline; local disk is at minimum an
  order of magnitude faster for this access pattern, and none of it needs to
  survive a runtime restart.

**Reference-photo matching removed.** `match_query_to_detections`,
`THRESHOLD`, and the `sklearn.cosine_similarity` import are gone from the
tracking cell — not just left unwired, deleted. The pipeline now
unconditionally profiles *every* person the tracker follows: `process_segments`
produces one segment per tracked person, and every segment goes straight into
Phase 2 for a natural-language description. `matches` was renamed to
`segments` throughout (`run_phase2`, `run_pipeline_for_video`, `eval_data`,
`main`) since it no longer means "people who matched a query photo."

## v1 — correctness, crashes, performance

## Crashes
1. **`run_phase2`** — `total=len(int(frame_paths/chunk_size))` raised `TypeError`
   on the first segment of the first video. Now `len(chunk_starts)`.
2. **Missing `import sys`** in the Ollama cell — it only worked because a *later*
   cell imported it. All imports are now local to the cell.
3. **`if box.id`** — falsy for a tensor holding `0`. Now `is not None`.
4. **Degenerate/out-of-bounds crops** — bboxes are clamped and zero-area boxes skipped.

## Silent correctness
5. **fps mismatch (the big one)** — frames extracted at 30fps, `detect_people`
   called with default `fps=1`, evaluation then multiplied by 30 again. Predicted
   frame numbers were 30× too large and matched nothing. A single `FPS = 30`
   constant is now threaded through extract → detect → segment → eval.
6. **`frame_%04d.jpg` → `frame_%06d.jpg`** — past 9999 frames (5.5 min at 30fps),
   `frame_10000.jpg` sorts *before* `frame_9999.jpg` and desyncs frame lookup.
   Frame paths now come from `result.path` rather than index-matching anyway.
7. **`track_id == -1`** detections were all lumped into one bogus segment. Dropped.
8. **False-negative counting** — was `len(events_df)`, charging the model for GT
   boxes on frames it never saw. Now restricted to *evaluable* GT.
9. **Dataset-wide `mean_iou`** — was rebuilt by replicating a rounded per-video
   mean. Raw per-box values are now pooled.
10. **CSV header drift** — append-mode writes now repair a stale header.
11. **`except Exception: print(e)`** hid the stack trace, which is why bug #1 looked
    like every video mysteriously failing. Now prints a traceback and a failure summary.
12. **Duplicate-detection double counting** — `evaluate_matches` originally let
    every prediction overlapping a GT box count as its own TP, so N boxes on
    one person scored N true positives against one annotation. Now COCO/VOC-style
    one-to-one assignment: highest-IoU prediction claims the box, later
    matches on the same box become `duplicate_detection` false positives.

## Performance
13. **`crop_people`** made a full-resolution `image.copy()` and wrote a full-size
    annotated JPEG *per detection*, not per frame. Fixed — and since nothing
    downstream reads `crop_path`, the whole step is now off by default
    (`SAVE_CROPS`/`SAVE_BOXED`).
14. **VLM frame subsampling** — the VLM was fed all 30fps frames (~450 calls for a
    single 2-minute track). Now sampled to `VLM_FPS = 1.0`, a ~30× reduction.
15. **`num_ctx` 12000 → 32768, `chunk_size` 18 → 8** — 18 images almost certainly
    overflowed 12k tokens, silently truncating the earliest frames.
16. **Model thrashing** — `run_phase2` alternated qwen ↔ mistral once per segment.
    Now two passes: all vision work, one unload, then all summarisation.
17. **`evaluate_matches`** was O(detections × GT rows) via a full-DataFrame scan
    plus `iterrows()` per prediction. GT is now indexed by frame once.
18. **Ollama pipe deadlock** — `ollama serve` was started with unread
    `subprocess.PIPE`; the server blocks once the 64KB buffer fills. Now `DEVNULL`.
19. **Health check** issued a real `chat()` call (forcing a 7B model load).
    Now hits `/api/tags`.


In [ ]:
# Install Ollama
!apt-get update -qq && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

OLLAMA_HOST = "http://127.0.0.1:11434"


def ollama_alive(timeout: float = 2.0) -> bool:
    """Cheap health check -- hits /api/tags, which does NOT load a model."""
    try:
        with urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=timeout):
            return True
    except Exception:
        return False


# FIX: stdout/stderr were subprocess.PIPE and nothing ever read them. Once
# ollama wrote ~64KB of logs the OS pipe buffer filled and the server blocked
# on write, hanging mid-run. DEVNULL discards the logs and never fills.
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# FIX: poll instead of a blind sleep(5).
for _ in range(30):
    if ollama_alive():
        print("Ollama server started")
        break
    time.sleep(1)
else:
    print("WARNING: Ollama server did not come up within 30s")


In [ ]:
# Pull Mistral (text model)
!ollama pull mistral:7b

# Pull Qwen VL (vision-language model)
!ollama pull qwen3-vl:8b   # or qwen2-vl, depending on what's available

In [ ]:
import subprocess
import os


def extract_frames(video_path: str, output_dir: str = "frames", fps: int = 1,
                   jpeg_quality: int = 2):
    """
    Extract frames from a video at the given fps rate.

    Args:
        video_path: path to the source video file
        output_dir: directory to write extracted frames into
        fps: frames per second to extract (1 = one frame every second)
        jpeg_quality: ffmpeg -q:v value (2 = high quality, 31 = worst)

    Returns:
        sorted list of frame filenames written to output_dir

    NOTE: filenames use %06d, not %04d. With %04d, a clip longer than 9999
    frames (5.5 min at 30fps) produces "frame_10000.jpg", which sorts BEFORE
    "frame_9999.jpg" lexicographically and silently desynchronises every
    index-based frame lookup downstream.

    NOTE: output frame_000001.jpg corresponds to SOURCE FRAME 0. Everything
    downstream uses 0-based frame indices to match VIRAT's annotations.
    """
    os.makedirs(output_dir, exist_ok=True)

    subprocess.run(
        [
            "ffmpeg", "-i", video_path,
            "-vf", f"fps={fps}",
            "-q:v", str(jpeg_quality),
            f"{output_dir}/frame_%06d.jpg",
            "-hide_banner", "-loglevel", "error",
        ],
        check=True,
    )

    frames = sorted(f for f in os.listdir(output_dir) if f.endswith(".jpg"))
    print(f"[extract_frames] Extracted {len(frames)} frames to '{output_dir}/' at {fps} fps")
    return frames


In [ ]:
!pip install ultralytics

In [ ]:
"""
Phase 1 - Facial Recognition & Tracking
Step 3: Person Detection (YOLOv8 / OpenCV)

For every extracted frame, detect all visible human entities and catalog
their spatial and temporal metadata: frame path, timestamp, bounding box
coordinates [x1, y1, x2, y2], track id, and detection confidence.
"""

import os
import re
from ultralytics import YOLO

PERSON_CLASS_ID = 0  # COCO class id for "person" in YOLOv8

_FRAME_NUM_RE = re.compile(r"(\d+)")

# FIX (perf): loading YOLO weights from disk was happening once PER VIDEO
# (detect_people constructed a fresh YOLO(model_name) every call). Weights
# don't change between videos, so this cache loads each named model once and
# reuses it for the rest of the run.
_MODEL_CACHE: dict[str, "YOLO"] = {}


def _get_yolo_model(model_name: str) -> "YOLO":
    if model_name not in _MODEL_CACHE:
        _MODEL_CACHE[model_name] = YOLO(model_name)
    return _MODEL_CACHE[model_name]


def _frame_index_from_path(path: str) -> int:
    """frame_000123.jpg -> 122 (0-based source frame index).

    ffmpeg numbers output files from 1, while VIRAT annotations are
    0-indexed, hence the -1.
    """
    m = _FRAME_NUM_RE.findall(os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse a frame number out of {path!r}")
    return int(m[-1]) - 1


def detect_people(frames_dir: str, fps: int = 1, conf_threshold: float = 0.5,
                  model_name: str = "yolov8n.pt", tracker: str = "botsort.yaml"):
    """
    Run YOLOv8 person detection + tracking on every frame in frames_dir.

    IMPORTANT: `fps` must be the SAME fps the frames were extracted at.
    Passing the default 1 while extracting at 30 makes every timestamp 30x
    too large, which silently invalidates all downstream segmentation and
    evaluation.

    Returns:
        list of dicts, one per detected person, with keys:
        frame_path, frame_idx, timestamp, track_id, bbox, confidence
    """
    model = _get_yolo_model(model_name)

    # FIX (correctness, now that the model is reused across videos):
    # persist=True carries tracker state across *separate* .track() calls on
    # the same model object. With a fresh model per video (the old behavior)
    # that was harmless -- there was never anything to persist. Now that the
    # model is cached and reused for every video in the dataset run,
    # persist=True would leak track state from the END of one video into the
    # START of the next unrelated video. persist=False resets the tracker at
    # the start of every call, which is what "independent videos" requires.
    #
    # Note this is unrelated to persistence *within* one call: a single
    # .track(source=frames_dir, stream=True) already walks every frame of
    # ONE video internally and maintains track continuity across them on its
    # own, regardless of this flag.
    results = model.track(
        source=frames_dir,
        persist=False,
        conf=conf_threshold,
        iou=0.5,
        classes=[PERSON_CLASS_ID],
        stream=True,
        tracker=tracker,
    )

    detections = []
    n_frames = 0

    for result in results:
        n_frames += 1

        # FIX: take the frame path from the result itself rather than
        # index-matching against a separately sorted os.listdir(). If YOLO
        # skips an unreadable image, index matching silently misattributes
        # every subsequent detection.
        frame_path = result.path
        frame_idx = _frame_index_from_path(frame_path)
        timestamp = frame_idx / fps  # seconds, 0-based

        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue

        # FIX (perf): the original called .item() / tensor-indexing once per
        # box per attribute (id, conf, each xyxy coordinate). Each of those
        # is a separate GPU->CPU sync point when boxes live on a CUDA
        # tensor. Pulling xyxy/conf/id to CPU as a single batched numpy array
        # per FRAME (not per box) cuts that down to 3 syncs per frame
        # regardless of how many people are in it, instead of ~6 per person.
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        ids = boxes.id.cpu().numpy() if boxes.id is not None else None

        for i in range(len(boxes)):
            # FIX: `if box.id` was falsy for a tensor holding 0. Compare to None.
            track_id = int(ids[i]) if ids is not None else -1
            x1, y1, x2, y2 = (int(v) for v in xyxy[i])
            detections.append({
                "frame_path": frame_path,
                "frame_idx": frame_idx,
                "timestamp": timestamp,
                "track_id": track_id,
                "bbox": [x1, y1, x2, y2],
                "confidence": round(float(confs[i]), 4),
            })

    print(f"[detect_people] Found {len(detections)} person detections across {n_frames} frames")
    return detections


In [ ]:
"""
Phase 1 - Facial Recognition & Tracking
Step 4: Spatial Target Cropping (OpenCV / Pillow)

Crops each detected person from their source frame and saves the crop
independently. Optionally saves ONE annotated copy of each full frame with
every bounding box on it.

NOTE: nothing downstream currently consumes "crop_path" or the boxed frames
-- run_phase2 feeds full frames to the VLM. This step is therefore optional.
Call it with save_crops=False, save_boxed=False (or skip it entirely) unless
you are inspecting output by eye.
"""

import os
import cv2


def crop_people(detections: list, crops_dir: str = "person_crops",
                boxed_dir: str = "boxed_frames",
                save_crops: bool = True, save_boxed: bool = False):
    """
    Crop every detected person from their source frame.

    Args:
        detections: list of dicts from detect_people
        crops_dir: output directory for individual person crops
        boxed_dir: output directory for full frames with boxes drawn
        save_crops: whether to write per-person crops at all
        save_boxed: whether to also save annotated full frames

    Returns:
        the same detections list, with a "crop_path" key added to each entry
        that was actually written
    """
    if not save_crops and not save_boxed:
        print("[crop_people] nothing to do (save_crops and save_boxed both False)")
        return detections

    if save_crops:
        os.makedirs(crops_dir, exist_ok=True)
    if save_boxed:
        os.makedirs(boxed_dir, exist_ok=True)

    # Group detections by frame so we read/decode each frame exactly once
    by_frame = {}
    for i, det in enumerate(detections):
        by_frame.setdefault(det["frame_path"], []).append(i)

    n_written = 0
    n_skipped = 0

    for frame_path, indices in by_frame.items():
        image = cv2.imread(frame_path)
        if image is None:
            print(f"[crop_people] WARNING: could not read {frame_path}")
            continue

        h, w = image.shape[:2]
        frame_name = os.path.splitext(os.path.basename(frame_path))[0]

        # FIX: one copy per FRAME, not one per detection. The original made a
        # full-resolution copy and wrote a full-size JPEG for every single
        # person in every single frame.
        boxed_image = image.copy() if save_boxed else None

        for idx in indices:
            x1, y1, x2, y2 = detections[idx]["bbox"]

            # FIX: clamp to image bounds and skip degenerate boxes. A
            # zero-area crop makes cv2.imwrite fail.
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(0, min(x2, w))
            y2 = max(0, min(y2, h))
            if x2 <= x1 or y2 <= y1:
                n_skipped += 1
                continue

            if save_boxed:
                cv2.rectangle(boxed_image, (x1, y1), (x2, y2), (0, 255, 0), 2)

            if save_crops:
                # FIX: name crops by track_id so the same person keeps the same
                # identifier across frames. The old per-frame counter restarted
                # at 1 every frame, so filenames were meaningless.
                track_id = detections[idx]["track_id"]
                tid = f"track_{track_id:03d}" if track_id >= 0 else "track_unk"
                crop_filename = f"{frame_name}_{tid}.jpg"
                crop_path = os.path.join(crops_dir, crop_filename)
                cv2.imwrite(crop_path, image[y1:y2, x1:x2])
                detections[idx]["crop_path"] = crop_path
                n_written += 1

        if save_boxed:
            cv2.imwrite(os.path.join(boxed_dir, f"{frame_name}.jpg"), boxed_image)

    print(f"[crop_people] Saved {n_written} crops to '{crops_dir}/' "
          f"({n_skipped} degenerate boxes skipped)")
    return detections


In [ ]:
"""
Phase 1 - Person Tracking & Temporal Segmentation

Groups every tracked person's detections into contiguous temporal segments.
There is no reference-photo / query-matching step: every person the tracker
picks up gets a segment, and every segment goes on to Phase 2 for a natural-
language activity description. This is intentional -- the pipeline profiles
*everyone* in the video, not a single subject matched against a photo.
"""


def process_segments(trajectory: list, max_gap_seconds: float = 2.0,
                     drop_untracked: bool = True,
                     min_duration_seconds: float = 0.0) -> list:
    """
    Group detections into contiguous per-track segments.

    A new segment starts when the track id changes, or when the gap between
    consecutive detections of the same track exceeds `max_gap_seconds`.

    Args:
        trajectory: detections from detect_people (timestamps in SECONDS)
        max_gap_seconds: dropout tolerance within a single track
        drop_untracked: discard detections the tracker never assigned an id to
        min_duration_seconds: drop segments shorter than this (end_time -
            start_time). A 1-2 frame track is almost always an ID flicker
            from the tracker, not a real activity, but it still costs a full
            VLM round trip in Phase 2 if it's allowed through. Default 0.0
            keeps the old behavior (no filtering). TRADEOFF: a genuinely
            brief real activity shorter than this threshold is also dropped
            and will show up as a miss in evaluation -- see the count this
            prints before raising it.

    Returns:
        list of segment dicts, one per contiguous (track_id, time-window) run,
        covering every person detected in the video.

    NOTE: `max_gap_seconds` is in seconds and only means that if timestamps
    are in seconds -- i.e. if detect_people was given the correct fps.
    """
    if not trajectory:
        return []

    if drop_untracked:
        n_before = len(trajectory)
        # Untracked detections all share track_id == -1; keeping them would
        # lump every stray, un-identity-linked box into one bogus "person".
        trajectory = [t for t in trajectory if t["track_id"] >= 0]
        n_dropped = n_before - len(trajectory)
        if n_dropped:
            print(f"[process_segments] dropped {n_dropped} untracked detections (track_id=-1)")
        if not trajectory:
            return []

    trajectory = sorted(trajectory, key=lambda x: (x["track_id"], x["timestamp"]))

    results = []
    segment_id = 1

    current_segment = [trajectory[0]]
    start_segment = trajectory[0]["timestamp"]
    prev_time = trajectory[0]["timestamp"]
    prev_track_id = trajectory[0]["track_id"]

    for traj in trajectory[1:]:
        track_changed = traj["track_id"] != prev_track_id
        time_gap = (traj["timestamp"] - prev_time) > max_gap_seconds

        if track_changed or time_gap:
            results.append({
                "segment_id": segment_id,
                "track_id": current_segment[0]["track_id"],
                "start_time": start_segment,
                "end_time": prev_time,
                "trajectory": current_segment,
            })
            segment_id += 1

            start_segment = traj["timestamp"]
            current_segment = [traj]
        else:
            current_segment.append(traj)

        prev_time = traj["timestamp"]
        prev_track_id = traj["track_id"]

    results.append({
        "segment_id": segment_id,
        "track_id": current_segment[0]["track_id"],
        "start_time": start_segment,
        "end_time": prev_time,
        "trajectory": current_segment,
    })

    if min_duration_seconds > 0:
        n_before = len(results)
        results = [s for s in results
                  if (s["end_time"] - s["start_time"]) >= min_duration_seconds]
        n_dropped = n_before - len(results)
        if n_dropped:
            print(f"[process_segments] dropped {n_dropped} segment(s) shorter than "
                  f"{min_duration_seconds}s (likely tracker noise, not real activity)")
        # Keep segment_id contiguous (1..N) after filtering rather than
        # leaving gaps where dropped segments used to be.
        for i, s in enumerate(results, start=1):
            s["segment_id"] = i

    print(f"[process_segments] {len(results)} segments across "
          f"{len({s['track_id'] for s in results})} distinct tracked people")

    return results


In [ ]:
!pip install ollama

In [ ]:
import json
import subprocess
import sys
import time

import cv2
from ollama import chat
from tqdm import tqdm

# NOTE: OLLAMA_HOST and ollama_alive() are defined in the earlier "start
# Ollama server" cell and reused here rather than redefined -- this notebook
# already relies on that cell-to-cell dependency chain everywhere (e.g.
# extract_frames, detect_people, process_segments are all defined upstream
# too), so this stays consistent with that instead of duplicating a second
# near-identical health check.

VLM_MODEL = "qwen3-vl:8b"
TEXT_MODEL = "mistral:7b"

# Qwen2.5-VL is a dynamic-resolution vision model: its token cost (and
# therefore its latency) scales with the pixel count of each image it's
# given, unlike YOLO which resizes to a fixed input size regardless of the
# source image. Frames straight out of ffmpeg are at the video's native
# resolution (often 1920x1080+); at chunk_size=8 that's 8 full-resolution
# images prefixed onto every single VLM call. Downscaling here cuts that
# cost directly. This ONLY affects what the vision model sees -- it does not
# touch the frames used for detection/tracking/bbox evaluation, so it has
# zero effect on bbox coordinates or ground-truth comparisons (see
# _load_vlm_image_bytes for why that separation matters).
VLM_IMAGE_MAX_WIDTH = 960

# Whether to crop each frame to (a padded region around) the segment's own
# track before sending it to the VLM, instead of the full scene.
#
# Added after finding, on a UCF-Crime "Arrest" video, that the VLM
# consistently failed to describe a real, human-confirmed altercation --
# even though 4 SEPARATE segments' full-frame VLM calls all covered the
# incident's time window. Not a track-selection problem (multiple
# independent calls saw the relevant frames and still missed it): the
# incident occupied roughly 1/16 of the frame area (confirmed by watching
# the source video), which at UCF-Crime's native resolution (320x240 in the
# clips checked) is on the order of 80x60 pixels -- too small for the VLM
# to resolve human-on-human contact in reliably.
VLM_USE_CROPS = True

# Padding is relative to the box's own size (not a fixed pixel margin),
# since person size in frame varies with distance from camera. 0.75 means
# each side is extended by 75% of that dimension -- generous specifically
# because a tight single-person crop risks cutting out the OTHER person(s)
# in the interaction (e.g. "surrounded by two people"), which would make
# the fix worse than the full-frame baseline for exactly the events it's
# meant to catch.
VLM_CROP_PAD_RATIO = 0.75

# Floor on crop size as a fraction of the full frame, so a small/distant
# detection still gets some surrounding context rather than an extremely
# tight box+padding.
VLM_CROP_MIN_FRACTION = 0.25

# Crops are naturally smaller than full frames; upscale (not just cap) so
# the VLM gets a comparable amount of effective pixel detail to work with,
# rather than sending it a tiny, low-information crop.
VLM_CROP_MIN_WIDTH = 480

# Whether to run a lightweight, class-unrestricted YOLO pass on each image
# actually sent to the VLM (post-crop) and surface any non-person classes
# found as a text hint in the prompt.
#
# Motivated by a UCF-Crime "Abuse" video where a black puppy, small and
# low-detail at native resolution (320x240), was consistently misdescribed
# by the VLM as "another individual in darker attire" -- the reference
# caption confirmed only one person was ever present. Reuses the already-
# cached YOLO model (_get_yolo_model/PERSON_CLASS_ID, from the person-
# detection cell) -- COCO includes "dog" and 78 other everyday classes,
# and YOLO's small-object detection tends to be more robust than a VLM's
# scene-level understanding for this kind of coarse "what class is this"
# question. This is a HINT, not a constraint: the hint prompt explicitly
# tells the model to verify against what it actually sees rather than
# trust the detector blindly, since a detector can also misfire.
#
# CAVEAT (found while diagnosing the case above): not every hallucinated
# "second individual" has a real detectable trigger. In the exact segment
# that motivated this, the dog wasn't even inside the crop sent to the
# VLM for that segment -- the model invented a coherent two-person
# narrative from an image of one person alone. This flag can only help
# when something real-but-ambiguous is actually in the image; it does
# nothing for pure confabulation from a single, clearly-visible subject.
VLM_DETECT_AUXILIARY_OBJECTS = True
VLM_AUXILIARY_CONF_THRESHOLD = 0.35


# NOTE (open-vocabulary rewrite): this prompt used to ask the model to
# check a fixed, numbered checklist of activities (originally VIRAT's 12
# event types; later 16 once DIVA-grounded violence/distress categories
# were added). That checklist caused two separate, confirmed problems on
# UCF-Crime: (1) a segment observed listing "acting aggressively",
# "physically fighting", "in visible distress", and "abandoning an object"
# as detected while its own summary text said none of them were present,
# with nothing in the underlying observations supporting any of them --
# see summarize_description()'s history for the structural cause (fixed
# there); (2) 6 of the (previously) 16 checklist items were VIRAT-specific
# vehicle activities (loading/unloading/trunk/getting in-out), which likely
# biased the model toward misreading ambiguous objects as vehicles even in
# scenes with no vehicle at all (confirmed: "a parked vehicle... rear
# door... loading" hallucinated in a bank-interior scene whose reference
# caption describes a table and a pager). More fundamentally, this
# pipeline's actual goal is recognizing a wide variety of activities, not
# just the ones on a fixed VIRAT/DIVA list -- constraining descriptions to
# a closed vocabulary works against that goal even when the vocabulary is
# well-chosen. Phase 2 now describes activities in open-ended natural
# language instead; CLASSES/the closed 12-class checklist is kept
# separately, for VIRAT-only benchmarking (see the VIRAT eval cell), not
# for description generation.
#
# NOTE (force/coercion wording, added after cropping fixed visibility but
# not interpretation): once VLM_USE_CROPS made person-to-person contact
# actually visible, the model still read a real, human-confirmed assault
# (a man pushed from behind, surrounded, dragged) as calm cooperation --
# "examining an object closely", "a cooperative effort or discussion". The
# paragraph below asks it to look for the specific visual evidence that
# distinguishes forced movement from cooperative movement, and explicitly
# names the failure mode observed (defaulting to a calm reading of
# sustained/close contact) so the model doesn't fall into it by default.
CHUNK_PROMPT = (
    "You are a video surveillance analyst. You are given a sequence of "
    "frames sampled over time from a single CCTV/surveillance video clip, "
    "in chronological order.\n\n"
    "Describe in detail what each visible person is doing throughout the "
    "sequence, IN YOUR OWN WORDS -- do not limit your description to any "
    "fixed list of activities or categories. Include their appearance, "
    "interactions with objects, vehicles, or buildings, any interactions "
    "with OTHER PEOPLE (physical contact, confrontation, pursuit, or "
    "cooperation), and any changes in their actions over time. Be as "
    "specific and concrete as possible -- name the objects involved and the "
    "nature of any contact between people, in chronological order.\n\n"
    "When two or more people are in physical contact or moving together, "
    "compare their posture and position ACROSS the frames in this sequence "
    "to judge whether the movement looks VOLUNTARY (walking together, "
    "cooperating, a calm exchange) or FORCED (one person pulling, pushing, "
    "dragging, restraining, or surrounding another against their will). "
    "Look specifically for: an off-balance or resisting posture, a person "
    "being moved by their arm/shoulder/clothing rather than moving under "
    "their own power, a body angled or displaced in a direction "
    "inconsistent with their own visible effort, multiple people converging "
    "on and enclosing one person, or visible tension/distress in posture or "
    "facial expression. Do NOT default to a calm or cooperative "
    "interpretation just because contact is sustained, repeated, or "
    "involves an object -- describe what the posture and motion actually "
    "show, even if that means describing struggle, resistance, or force."
)

# NOTE (open-vocabulary rewrite): this used to also ask the model to pick
# activities from the same fixed checklist as CHUNK_PROMPT, constrained via
# SUMMARY_FORMAT_SCHEMA's enum -- removed for the same reasons documented
# above CHUNK_PROMPT (checklist-induced misclassification, and it working
# against this pipeline's actual goal of recognizing a wide variety of
# activities, not just a fixed list). This prompt now only asks for a
# chronological natural-language summary; SUMMARY_FORMAT_SCHEMA below has
# been simplified to match (just a "summary" string, no "activity" array).
#
# FIX (historical, now moot but worth keeping as a lesson): this prompt's
# instructions used to describe a free-text "Summary: ...\nDetected
# Activities:\n- ..." bullet format that directly contradicted the JSON
# object response_format actually forced on the output -- the model's own
# instructions and what it was forced to emit disagreed with each other.
# That mismatch is moot now that there's only one field to return, but it's
# also why this prompt describes its JSON output directly rather than in
# free-text-format language, matching the style already used by
# RISK_PROMPT_TEMPLATE/SUMMARY_EVAL_PROMPT_TEMPLATE/
# REFERENCE_EVAL_PROMPT_TEMPLATE.
SUMMARY_PROMPT_TEMPLATE = (
    "Given the following observations from different portions of the video:\n\n"
    "{combined_results}\n\n"
    "Write a single, chronological, detailed natural-language summary of "
    "everything that happened, in your own words. Do not limit yourself to "
    "any fixed list of activities or categories -- describe what actually "
    "happened, as specifically and concretely as possible, including any "
    "interactions between people and how they appeared to unfold.\n\n"
    "Return a JSON object with exactly one field:\n"
    "- summary: a chronological, detailed natural-language description of "
    "the events, as a string."
)

# NOTE (VIRAT-only, as of the open-vocabulary rewrite): Phase 2 generation
# (CHUNK_PROMPT/SUMMARY_PROMPT_TEMPLATE/SUMMARY_FORMAT_SCHEMA above) no
# longer references this list at all -- kept only because the VIRAT eval
# cell still uses it independently, to convert VIRAT's own numeric
# event_type ground truth into readable names. If you're not running the
# VIRAT eval cell, this list isn't used anywhere.
CLASSES = [
    # 0-11: VIRAT's original 12 event types, in VIRAT's own order --
    # evaluate_matches()/eval_data() (VIRAT cell) index into this list as
    # CLASSES[event_type - 1], so this order must not change.
    "Person loading an object into a vehicle",
    "Person unloading an object from a vehicle",
    "Person opening a vehicle trunk",
    "Person closing a vehicle trunk",
    "Person getting into a vehicle",
    "Person getting out of a vehicle",
    "Person gesturing",
    "Person digging",
    "Person carrying an object",
    "Person running",
    "Person entering a facility",
    "Person exiting a facility",
    # 12-15: added for UCF-Crime -- worded to match DIVA's own activity
    # definitions (Aggressive/Fighting/Distress/Abandoning Object) rather
    # than invented language. VIRAT ground truth never has event_type > 12,
    # so appending here doesn't disturb the VIRAT-side indexing above.
    "Person acting aggressively toward another person",
    "Person physically fighting with another person",
    "Person in visible distress",
    "Person abandoning an object",
]

# Simplified to a single free-text field -- see the NOTE above
# SUMMARY_PROMPT_TEMPLATE for why the "activity" enum was removed. CLASSES
# itself is kept (not deleted) because the VIRAT eval cell still uses it
# independently, for converting VIRAT's own numeric event_type ground truth
# into readable names -- that usage has nothing to do with what Phase 2
# generates.
SUMMARY_FORMAT_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {
            "type": "string"
        }
    },
    "required": ["summary"],
}


# ---------------------------------------------------------------------------
# Risk assessment (Phase 2, part 1 of the objective: "suspicious" classification)
#
# summarize_description() above answers "what happened" (activity labels
# benchmarkable against VIRAT + a free-text description). Nothing upstream
# of this point ever answers "should this be flagged" -- that's a genuinely
# separate judgment call, not a relabeling of the VIRAT classes: e.g.
# "Person opening a vehicle trunk" is not on its own suspicious, but the
# same action paired with loitering beforehand or concealment would be.
# Kept as its own pass (own prompt, own call) rather than folded into
# summarize_description so a bad risk judgment can't corrupt the
# VIRAT-benchmarkable activity/summary output, and vice versa.
# ---------------------------------------------------------------------------

# NOTE: no longer takes a separate "activities" field -- Phase 2 dropped
# the fixed-checklist activity classification (see CHUNK_PROMPT/
# SUMMARY_PROMPT_TEMPLATE), so the free-text description is now the only
# evidence assess_risk has, which is also the more information-rich signal
# a canned category label ever was.
RISK_PROMPT_TEMPLATE = (
    "You are a security analyst reviewing an automated description of one "
    "person's behavior in a video, produced by an activity-recognition "
    "system. Base your assessment ONLY on the evidence in the description "
    "below -- do not assume malicious intent from ordinary behavior. For "
    "example, loading groceries into a car is not suspicious; the SAME "
    "action performed while avoiding cameras, at an unusual hour, or paired "
    "with forced entry would be.\n\n"
    "Description:\n{summary}\n\n"
    "Assess this person's behavior for concrete security-relevant risk "
    "indicators such as: unauthorized access or entry, concealment or "
    "evasive behavior, aggression or threatening gestures, tampering with a "
    "vehicle or property that appears unauthorized, or other behavior a "
    "human security analyst would flag for closer review.\n\n"
    "Return:\n"
    "- suspicious: true only if the description contains a CONCRETE "
    "indicator of risk, not just an activity that sounds notable on its "
    "own.\n"
    "- risk_score: an integer 0-100, where 0 is entirely benign/routine "
    "behavior and 100 is an unambiguous, severe threat indicator.\n"
    "- rationale: one or two sentences citing the SPECIFIC evidence from "
    "the description that drove the score."
)

RISK_FORMAT_SCHEMA = {
    "type": "object",
    "properties": {
        "suspicious": {"type": "boolean"},
        "risk_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "rationale": {"type": "string"},
    },
    "required": ["suspicious", "risk_score", "rationale"],
}


# How many times to retry a failed chat() call before giving up on a chunk.
MAX_RETRIES = 5
# Base delay (seconds) for exponential backoff between retries.
RETRY_BASE_DELAY = 5

# FIX (perf/redundancy): the original unconditionally evicted the vision
# model (keep_alive=0) after every video's vision pass, then had to pay a
# full reload for the next video's vision pass. That's the right call on a
# small-VRAM GPU (e.g. T4/L4) where two 7B models don't comfortably coreside.
# This notebook's Colab runtime is configured for an A100 (see notebook
# metadata), which has enough VRAM for both quantized 7B models to stay
# resident simultaneously -- so evicting between passes is pure overhead:
# N extra reloads across an N-video run for no memory benefit. Flip this to
# True if you switch to a smaller GPU and start seeing OOM/thrash.
UNLOAD_VLM_BETWEEN_PASSES = False


def ensure_ollama_running(model: str = TEXT_MODEL) -> None:
    """Make sure the Ollama server is up; (re)start it if not.

    Safe to call frequently -- it only acts if the server looks dead.
    """
    if ollama_alive():
        return

    print("[ollama] not responding, attempting to (re)start server...", file=sys.stderr)
    try:
        # FIX: DEVNULL, not PIPE. An unread PIPE fills its 64KB buffer and
        # deadlocks the server.
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    except FileNotFoundError:
        print("[ollama] 'ollama' binary not found on PATH -- cannot auto-restart.", file=sys.stderr)
        return

    for _ in range(15):
        time.sleep(2)
        if ollama_alive():
            print("[ollama] server back up.", file=sys.stderr)
            return
    print("[ollama] server still not responding after restart attempt.", file=sys.stderr)


def chat_with_retry(*, model: str, messages: list, options: dict | None = None,
                    response_format: dict | None = None, keep_alive: str = "45m") -> dict:
    """Wrapper around ollama.chat() that retries on transient failures
    (connection drops, server not-yet-up, etc.) with exponential backoff,
    restarting the Ollama server if it appears to be down."""
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            kwargs = {"model": model, "messages": messages, "keep_alive": keep_alive}
            if options is not None:
                kwargs["options"] = options
            if response_format is not None:
                kwargs["format"] = response_format
            return chat(**kwargs)
        except Exception as e:
            last_exc = e
            print(f"[ollama] call failed (attempt {attempt}/{MAX_RETRIES}): {e}", file=sys.stderr)
            ensure_ollama_running(model=model)
            if attempt < MAX_RETRIES:
                delay = RETRY_BASE_DELAY * (2 ** (attempt - 1))
                time.sleep(delay)
    raise RuntimeError(f"Ollama call failed after {MAX_RETRIES} attempts") from last_exc


def unload_model(model: str) -> None:
    """Ask Ollama to evict a model from VRAM (keep_alive=0).

    Called between the vision pass and the text pass so the two 7B models
    aren't fighting for the same GPU memory.
    """
    try:
        chat(model=model, messages=[{"role": "user", "content": "bye"}],
             options={"num_predict": 1}, keep_alive=0)
    except Exception as e:
        print(f"[ollama] could not unload {model}: {e}", file=sys.stderr)


def _compute_crop_box(bbox: list[int], img_w: int, img_h: int,
                     pad_ratio: float = VLM_CROP_PAD_RATIO,
                     min_fraction: float = VLM_CROP_MIN_FRACTION) -> tuple[int, int, int, int]:
    """Pad a detection box for VLM framing: enough to likely keep nearby
    people the track is interacting with in frame, with a floor so a small
    or distant detection still gets meaningful surrounding context.

    Padding is proportional to the box's own size, not a fixed pixel
    margin -- a person near the camera and one far away need very
    different absolute padding to get comparable context.
    """
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    pad_w, pad_h = bw * pad_ratio, bh * pad_ratio

    cx1, cy1 = x1 - pad_w, y1 - pad_h
    cx2, cy2 = x2 + pad_w, y2 + pad_h

    min_w, min_h = img_w * min_fraction, img_h * min_fraction
    if (cx2 - cx1) < min_w:
        cx_c = (cx1 + cx2) / 2
        cx1, cx2 = cx_c - min_w / 2, cx_c + min_w / 2
    if (cy2 - cy1) < min_h:
        cy_c = (cy1 + cy2) / 2
        cy1, cy2 = cy_c - min_h / 2, cy_c + min_h / 2

    cx1 = max(0, min(round(cx1), img_w - 1))
    cy1 = max(0, min(round(cy1), img_h - 1))
    cx2 = max(cx1 + 1, min(round(cx2), img_w))
    cy2 = max(cy1 + 1, min(round(cy2), img_h))
    return cx1, cy1, cx2, cy2


def _detect_auxiliary_objects(img, model_name: str = "yolov8n.pt",
                              conf_threshold: float = VLM_AUXILIARY_CONF_THRESHOLD,
                              exclude_class_ids: tuple[int, ...] = (PERSON_CLASS_ID,)) -> list[str]:
    """Run YOLO (class-unrestricted) on an already-loaded image array --
    e.g. the exact crop about to be sent to the VLM -- and return the names
    of any non-person classes found, as a grounding hint.

    Not tracking, no temporal continuity, no confidence bar as strict as
    detect_people's (0.5): this is advisory text for the prompt, not a
    structural detection with downstream consequences, so a lower threshold
    trades some false positives for fewer missed small/ambiguous objects --
    acceptable since the hint is phrased for the model to verify, not trust
    outright (see process_chunk).
    """
    model = _get_yolo_model(model_name)
    results = model(img, verbose=False)[0]
    names = []
    if results.boxes is not None:
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            if cls_id in exclude_class_ids or conf < conf_threshold:
                continue
            names.append(model.names[cls_id])
    return sorted(set(names))


def _load_vlm_image_bytes(path: str, bbox: list[int] | None = None,
                          max_width: int = VLM_IMAGE_MAX_WIDTH) -> tuple[bytes, list[str]]:
    """Read a frame and re-encode it, purely for VLM input: optionally
    cropped to (a padded region around) `bbox`, then downscaled if still
    over `max_width` or upscaled if under VLM_CROP_MIN_WIDTH. Also runs
    _detect_auxiliary_objects on the FINAL image (post-crop, exactly what
    the VLM will see) if VLM_DETECT_AUXILIARY_OBJECTS is on, so any hint
    surfaced is grounded in the same pixels the model is looking at, not a
    different view of the frame.

    Deliberately separate from anything detection/tracking/eval touches.
    Resizing/cropping at EXTRACTION time (in extract_frames) would put
    every downstream bbox in a different coordinate space than VIRAT's
    native-resolution ground truth, silently corrupting every IoU
    comparison -- exactly the kind of bug fixed elsewhere in this notebook
    (the fps mismatch). Doing it only on the copy handed to the VLM
    sidesteps that risk entirely: detection still runs on full-resolution,
    full-frame images, and this function's output is never looked at by
    evaluate_matches.

    Returns (jpeg_bytes, auxiliary_object_names).
    """
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"[_load_vlm_image_bytes] could not read {path}")

    h, w = img.shape[:2]

    if bbox is not None and VLM_USE_CROPS:
        x1, y1, x2, y2 = _compute_crop_box(bbox, w, h)
        img = img[y1:y2, x1:x2]
        h, w = img.shape[:2]
        if w < VLM_CROP_MIN_WIDTH:
            scale = VLM_CROP_MIN_WIDTH / w
            img = cv2.resize(img, (VLM_CROP_MIN_WIDTH, max(1, round(h * scale))),
                             interpolation=cv2.INTER_CUBIC)
            h, w = img.shape[:2]

    if w > max_width:
        scale = max_width / w
        img = cv2.resize(img, (max_width, max(1, round(h * scale))),
                         interpolation=cv2.INTER_AREA)

    aux_objects = _detect_auxiliary_objects(img) if VLM_DETECT_AUXILIARY_OBJECTS else []

    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])
    if not ok:
        raise RuntimeError(f"[_load_vlm_image_bytes] failed to encode {path}")
    return buf.tobytes(), aux_objects


def process_chunk(chunk: list[tuple[str, list[int]]], model: str = VLM_MODEL,
                  num_ctx: int = 32768) -> str:
    """Send one chunk of (frame_path, bbox) pairs to the VLM and return its
    free-text description of the activity in that chunk.

    Each pair is the segment's own frame + the track's detection box in
    that frame -- used to crop (see VLM_USE_CROPS) so the VLM sees the
    tracked person at higher effective resolution than the full scene.
    Also appends a grounding hint listing any non-person object classes
    YOLO found across this chunk's (post-crop) images, if
    VLM_DETECT_AUXILIARY_OBJECTS is on and it found anything -- see
    _detect_auxiliary_objects for why, and its docstring's caveat that this
    only helps when something real is actually in frame.

    NOTE ON num_ctx: this was 12000 with chunk_size=18. Qwen2.5-VL turns a
    single image into hundreds-to-1000+ tokens depending on resolution, so
    18 images routinely blew past 12k and the earliest frames were silently
    truncated away -- which defeats the whole "chronological sequence"
    premise of the prompt. Keep chunk_size * per-image tokens comfortably
    under num_ctx. Downscaling images (VLM_IMAGE_MAX_WIDTH) makes this
    margin considerably more comfortable at the same chunk_size.
    """
    loaded = [_load_vlm_image_bytes(p, bbox=bbox) for p, bbox in chunk]
    images = [img_bytes for img_bytes, _ in loaded]
    aux_objects = sorted({name for _, names in loaded for name in names})

    prompt = CHUNK_PROMPT
    if aux_objects:
        prompt += (
            "\n\nNote: a separate object detector flagged the following "
            "additional object classes as possibly present somewhere in "
            "these images: " + ", ".join(aux_objects) + ". This is a hint, "
            "not a fact -- verify against what you actually see in the "
            "images before mentioning it, and ignore it if you can't "
            "correlate it with anything visible."
        )

    response = chat_with_retry(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
                "images": images,
            }
        ],
        options={"num_ctx": num_ctx},
    )
    return response["message"]["content"]


def summarize_description(combined_results: str, model: str = TEXT_MODEL) -> dict:
    """Summarize all chunk descriptions for one segment and return a
    {"summary": str} dict."""
    response = chat_with_retry(
        model=model,
        messages=[
            {
                "role": "user",
                "content": SUMMARY_PROMPT_TEMPLATE.format(combined_results=combined_results),
            }
        ],
        response_format=SUMMARY_FORMAT_SCHEMA,
    )
    return json.loads(response["message"]["content"])


def assess_risk(summary: str, model: str = TEXT_MODEL) -> dict:
    """Score one segment's already-generated description for suspicious /
    threat-relevant behavior.

    Text-only, over summarize_description's output -- does not re-touch the
    frames or the VLM. Returns {"suspicious": bool, "risk_score": 0-100,
    "rationale": str}. An empty summary (segment produced no chunk
    descriptions) short-circuits to a benign default rather than paying a
    model call for no evidence.
    """
    if not summary:
        return {
            "suspicious": False,
            "risk_score": 0,
            "rationale": "No description was generated for this segment.",
        }

    response = chat_with_retry(
        model=model,
        messages=[
            {
                "role": "user",
                "content": RISK_PROMPT_TEMPLATE.format(summary=summary),
            }
        ],
        response_format=RISK_FORMAT_SCHEMA,
    )
    return json.loads(response["message"]["content"])


def subsample(items: list, source_fps: float, target_fps: float) -> list:
    """Keep roughly `target_fps` frames per second out of a `source_fps`
    list -- generic over the list's element type (plain frame paths, or
    (frame_path, bbox) pairs), since it only ever slices, never inspects
    elements.

    Detection/tracking wants every frame for tight boxes; the VLM does not.
    At 30fps a two-minute track is ~3600 frames, which is ~450 VLM calls for
    ONE segment. Sampling to 1fps cuts that by 30x with no meaningful loss
    for activity recognition.
    """
    if target_fps <= 0 or target_fps >= source_fps:
        return items
    stride = max(1, int(round(source_fps / target_fps)))
    return items[::stride]


def run_phase2(segments, chunk_size: int = 8, source_fps: float = 30.0,
               vlm_fps: float = 1.0, max_chunks_per_segment: int | None = None):
    """Run the two-stage VLM pipeline over every tracked person's segment.

    There is no reference-photo filtering here -- `segments` is every
    contiguous per-track segment process_segments() produced, i.e. every
    person the tracker picked up in the video. Each one gets its own
    open-vocabulary natural-language description (no fixed activity
    checklist -- see CHUNK_PROMPT/SUMMARY_PROMPT_TEMPLATE).

    Pass 1 describes every chunk of every segment with the VISION model.
    Pass 2 summarizes each segment with the TEXT model.
    Pass 3 scores each segment's summary for suspicious / threat-relevant
    behavior with the TEXT model (see assess_risk).

    FIX (structural): the original interleaved qwen -> mistral -> qwen ->
    mistral once per segment. On GPUs where both 7B models can't coreside,
    every switch pays a model reload despite keep_alive. Doing all the
    vision work first, then all the text work, means at most one swap for
    the whole run instead of one per segment (see UNLOAD_VLM_BETWEEN_PASSES
    for whether that one swap happens at all on this runtime).

    Returns a list of per-segment results with "summary" (free-text
    description), "suspicious" (bool), "risk_score" (0-100 int), and
    "risk_rationale" (str) from the risk-assessment pass. No "activities"
    key -- that was Phase 2's old fixed-checklist classification, now
    removed (see the NOTE above CHUNK_PROMPT).
    """
    # ---------- Pass 1: vision ----------
    per_segment_chunks = []
    n_vision_calls = 0
    vision_time = 0.0

    for segment in tqdm(segments, desc="VLM: describing segments"):
        # (frame_path, bbox) pairs -- bbox is this track's own detection box
        # in that frame, used for cropping (see VLM_USE_CROPS /
        # _load_vlm_image_bytes). Kept together through subsampling and
        # chunking so process_chunk always knows which box goes with which
        # frame.
        frame_items = [(step["frame_path"], step["bbox"]) for step in segment["trajectory"]]
        frame_items = subsample(frame_items, source_fps, vlm_fps)

        chunk_starts = list(range(0, len(frame_items), chunk_size))
        if max_chunks_per_segment is not None:
            chunk_starts = chunk_starts[:max_chunks_per_segment]

        # FIX: this line was
        #     total=len(int(frame_paths/chunk_size))
        # which raises TypeError (list / int) on the very first segment. The
        # bare `except Exception` in main() then swallowed it, so every video
        # "failed" with a cryptic message and no inference JSON was written.
        descriptions = []
        for i in tqdm(chunk_starts, total=len(chunk_starts),
                      desc=f"seg {segment['segment_id']}", leave=False):
            chunk = frame_items[i:i + chunk_size]
            t0 = time.time()
            descriptions.append(process_chunk(chunk))
            vision_time += time.time() - t0
            n_vision_calls += 1

        per_segment_chunks.append((segment, descriptions))

    # Free the VLM before loading the text model -- only if configured to
    # (see UNLOAD_VLM_BETWEEN_PASSES above). On this notebook's A100 runtime
    # both models fit in VRAM at once, so the default is to leave it loaded.
    if UNLOAD_VLM_BETWEEN_PASSES:
        unload_model(VLM_MODEL)

    # ---------- Pass 2: text summarisation ----------
    segment_results = []
    n_text_calls = 0
    text_time = 0.0

    for segment, chunk_descriptions in tqdm(per_segment_chunks, desc="LLM: summarizing segments"):
        if chunk_descriptions:
            combined_results = "\n".join(chunk_descriptions)
            t0 = time.time()
            summary = summarize_description(combined_results)
            text_time += time.time() - t0
            n_text_calls += 1
        else:
            summary = {"summary": ""}

        segment_results.append({
            "segment_id": segment["segment_id"],
            "track_id": segment["track_id"],
            "start_time": segment["start_time"],
            "end_time": segment["end_time"],
            "descriptions": chunk_descriptions,
            "summary": summary["summary"],
        })

    # ---------- Pass 3: risk assessment ----------
    # Same reload-avoidance reasoning as the vision -> text split above: all
    # of pass 2's summaries are ready before pass 3 starts, so this is (at
    # most) one more model swap for the whole run, not one per segment.
    n_risk_calls = 0
    risk_time = 0.0

    for result in tqdm(segment_results, desc="LLM: assessing risk"):
        t0 = time.time()
        risk = assess_risk(result["summary"])
        risk_time += time.time() - t0
        n_risk_calls += 1

        result["suspicious"] = risk["suspicious"]
        result["risk_score"] = risk["risk_score"]
        result["risk_rationale"] = risk["rationale"]

    # This is the single most useful diagnostic in the whole pipeline for
    # "why is this slow": a healthy GPU call to a 7B model with 8 downscaled
    # images typically lands in the low single-digit seconds. If avg call
    # time below is 20-60s+, that points at CPU inference (check GPU status
    # printed at the top of the pipeline cell) rather than anything tunable
    # here.
    avg_vision = vision_time / n_vision_calls if n_vision_calls else 0.0
    avg_text = text_time / n_text_calls if n_text_calls else 0.0
    avg_risk = risk_time / n_risk_calls if n_risk_calls else 0.0
    print(
        f"[run_phase2 timing] vision: {n_vision_calls} calls, "
        f"{vision_time:.1f}s total, {avg_vision:.1f}s/call avg  |  "
        f"text: {n_text_calls} calls, {text_time:.1f}s total, {avg_text:.1f}s/call avg  |  "
        f"risk: {n_risk_calls} calls, {risk_time:.1f}s total, {avg_risk:.1f}s/call avg"
    )

    return segment_results


In [ ]:
"""
Phase 2 -- shared, dataset-agnostic pipeline driver.

Everything in this cell behaves the same regardless of which dataset a
video comes from: local scratch dirs, a GPU status check, the copy/clear
helpers, the per-video pipeline driver (extract -> detect+track -> segment
-> phase2 -> risk rollup), a generic CSV-append helper, and a generic
precision/recall/F1/accuracy helper.

Both the VIRAT eval cell and the UCF-Crime eval cell depend on this cell
(plus the VLM-pipeline cell before it, for CLASSES/chat_with_retry/
TEXT_MODEL/run_phase2) -- NEITHER depends on the other, so either dataset
can be run without the other's cell ever executing.

VIDEO_DIR and INFERENCE_DIR are declared here as placeholders; the VIRAT
cell and the UCF-Crime cell each assign their own dataset paths into these
same names before calling run_pipeline_for_video. Whichever dataset cell
ran most recently "owns" them -- don't interleave runs of both datasets
without re-setting paths in the one you're switching back to.
"""

import csv
import os
import shutil
import subprocess
import sys
import time
import traceback

import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------------------------
# ONE fps for the whole pipeline.
#
# This was THE correctness bug: frames were extracted at 30fps, detect_people
# was called with its default fps=1 (so "timestamps" were really frame
# indices), and evaluation then multiplied those by 30 again. Predicted frame
# numbers came out 30x too large and matched essentially nothing.
#
# Lower FPS to trade box-level temporal resolution for speed and disk.
# ---------------------------------------------------------------------------
FPS = 30

# Frames per second actually shown to the VLM (see subsample()).
VLM_FPS = 1.0
VLM_CHUNK_SIZE = 8

# Segments shorter than this are almost always tracker noise (a person
# detected for 1-2 frames as an ID flickers in and out) rather than a real
# activity -- but they still cost at least one full VLM round trip each.
# Dropped before Phase 2 ever sees them. TRADEOFF: a genuinely brief real
# activity shorter than this also gets skipped and counted as a miss in
# eval; 0.5s (15 frames at 30fps) is deliberately conservative -- raise it
# only after checking how many segments it's actually dropping (printed by
# process_segments) and spot-checking a few inference JSONs.
MIN_SEGMENT_DURATION = 0.5

# The crop / boxed-frame step is not consumed by anything downstream.
# Turn on only for visual inspection.
SAVE_CROPS = False
SAVE_BOXED = False

# Local scratch (ephemeral, rewritten per video by clear_pipeline_dirs()).
# Not dataset-specific -- both VIRAT and UCF-Crime videos get copied here,
# extracted here, and cleared from here between videos.
WORK_DIR = "/content/work"
FRAMES_DIR = os.path.join(WORK_DIR, "frames")
BOXED_DIR = os.path.join(WORK_DIR, "boxed_frames")
CROPS_DIR = os.path.join(WORK_DIR, "person_crops")
os.makedirs(WORK_DIR, exist_ok=True)

# Placeholders -- the VIRAT cell or the UCF-Crime cell assigns these to its
# own dataset paths before calling run_pipeline_for_video. Deliberately not
# set here: this cell has no opinion on which dataset you're running.
VIDEO_DIR = None
INFERENCE_DIR = None


def _print_gpu_status():
    """Print whether a GPU is actually attached and how much VRAM is in use.

    If this prints "no GPU detected", that is almost certainly the entire
    explanation for an hour-long single video: YOLO on CPU and two 7B models
    on CPU through Ollama are each roughly one to two orders of magnitude
    slower than on any Colab GPU tier. Check Runtime > Change runtime type >
    Hardware accelerator before looking at anything else below.
    """
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        )
        if out.returncode == 0 and out.stdout.strip():
            print("GPU status:                ", out.stdout.strip())
        else:
            print("GPU status:                 nvidia-smi ran but returned nothing --"
                  " no GPU detected. Check Runtime > Change runtime type > GPU.")
    except FileNotFoundError:
        print("GPU status:                 no GPU detected (nvidia-smi not found)."
              " Check Runtime > Change runtime type > GPU.")
    except Exception as e:
        print(f"GPU status:                 could not check ({e})")



def clear_pipeline_dirs():
    """Removes stale ephemeral per-video output dirs so each video starts
    clean. These live under WORK_DIR (local disk), never on Drive.

    Also sweeps any leftover local video copy from a run that crashed
    between _copy_video_locally() and the os.remove() cleanup after
    extraction -- otherwise those accumulate across a multi-video run.
    """
    for d in (FRAMES_DIR, BOXED_DIR, CROPS_DIR):
        if os.path.exists(d):
            shutil.rmtree(d)
    for f in os.listdir(WORK_DIR):
        if f.lower().endswith((".mp4", ".avi", ".mov")):
            try:
                os.remove(os.path.join(WORK_DIR, f))
            except OSError:
                pass



def append_metrics_row(path: str, metrics: dict) -> None:
    """Append one row to a CSV, repairing the header if the schema changed.

    FIX: the original opened in "a" mode and only wrote a header when the
    file was absent. Re-running after changing the metric set produced rows
    silently misaligned with a stale header.

    FIX (perf): the original read and parsed the ENTIRE existing CSV with
    DictReader on every single call just to compare headers -- O(file size)
    work per video, even though the header only changes if you edit the
    metrics schema. This reads just the first line for that check; the full
    read+rewrite only happens on the rare path where the header actually
    changed.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fieldnames = list(metrics.keys())
    file_exists = os.path.exists(path)

    if file_exists:
        with open(path, "r", newline="") as f:
            first_line = f.readline()
        existing_header = next(csv.reader([first_line]), [])

        if existing_header and existing_header != fieldnames:
            # Schema changed since the last run -- repair by reading the
            # full file once and rewriting with a unioned header. This is
            # the only path that pays O(file size); it only triggers when
            # the metrics dict's keys actually changed.
            with open(path, "r", newline="") as f:
                existing_rows = list(csv.DictReader(f))
            fieldnames = list(dict.fromkeys(existing_header + fieldnames))
            with open(path, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames, restval="")
                writer.writeheader()
                writer.writerows(existing_rows)
                writer.writerow(metrics)
            return

    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, restval="")
        if not file_exists:
            writer.writeheader()
        writer.writerow(metrics)


def _copy_video_locally(src_path: str, dst_dir: str, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    """Copy a video from Drive to local disk with a visible byte-progress bar.

    ffmpeg reading a multi-hundred-MB file directly off a Drive FUSE mount is
    the slowest part of this pipeline and gives NO progress output while it
    runs -- a working-but-slow extraction and a genuine hang look identical.
    Copying locally first (a) is faster because local disk beats FUSE for
    sequential reads, and (b) is visible, because tqdm updates per chunk
    instead of going silent until the whole ffmpeg call returns.
    """
    os.makedirs(dst_dir, exist_ok=True)
    dst_path = os.path.join(dst_dir, os.path.basename(src_path))
    size = os.path.getsize(src_path)

    with open(src_path, "rb") as fsrc, open(dst_path, "wb") as fdst, tqdm(
        total=size, unit="B", unit_scale=True,
        desc=f"copying {os.path.basename(src_path)} from Drive",
    ) as pbar:
        while True:
            chunk = fsrc.read(chunk_bytes)
            if not chunk:
                break
            fdst.write(chunk)
            pbar.update(len(chunk))

    return dst_path


def run_pipeline_for_video(video_file_name: str, fps: int = FPS):
    """
    Runs the detect -> (crop) -> track/segment -> phase2 pipeline for one
    video and returns (segments, phase2_results) for downstream evaluation.

    No reference photo is involved anywhere in this pipeline: every person
    the tracker follows gets a segment, and every segment gets described.

    Prints a wall-clock timing breakdown for every stage. This is the
    fastest way to find out WHERE an unexpectedly slow run is actually
    spending its time instead of guessing -- e.g. if [4/4] dominates, the
    VLM is the bottleneck (check segment count and MIN_SEGMENT_DURATION);
    if [3/4] dominates on a video with thousands of frames, check GPU status
    above; if [1/4] dominates, Drive throughput is the issue.
    """
    video_basename = os.path.splitext(video_file_name)[0]
    drive_video_path = os.path.join(VIDEO_DIR, video_file_name)

    print(f"\n=== {video_file_name} ===")

    clear_pipeline_dirs()

    t0 = time.time()
    print("[1/4] copying video from Drive to local disk...")
    local_video_path = _copy_video_locally(drive_video_path, WORK_DIR)
    t1 = time.time()

    print("[2/4] extracting frames (ffmpeg, local read)...")
    extract_frames(local_video_path, FRAMES_DIR, fps=fps)
    t2 = time.time()

    # Raw video isn't needed past extraction -- free the local disk space
    # before the next video's copy (and before any large frame/crop writes).
    try:
        os.remove(local_video_path)
    except OSError:
        pass

    # FIX: fps is passed explicitly. The default of 1 here was the source of
    # the 30x timestamp error.
    print("[3/4] running person detection + tracking (YOLO)...")
    detections = detect_people(FRAMES_DIR, fps=fps)
    print("detections:", len(detections))
    t3 = time.time()

    if SAVE_CROPS or SAVE_BOXED:
        detections = crop_people(detections, crops_dir=CROPS_DIR, boxed_dir=BOXED_DIR,
                                 save_crops=SAVE_CROPS, save_boxed=SAVE_BOXED)

    # Every tracked person becomes a segment -- there is no query-photo
    # filter here, this is the full population of people in the video.
    segments = process_segments(detections, min_duration_seconds=MIN_SEGMENT_DURATION)
    print("segments:", len(segments),
          "trajectory points:", sum(len(s["trajectory"]) for s in segments))

    print("[4/4] describing each person's segments (VLM + summarizer)...")
    results = run_phase2(
        segments,
        chunk_size=VLM_CHUNK_SIZE,
        source_fps=fps,
        vlm_fps=VLM_FPS,
    )
    t4 = time.time()

    # Inference JSON is a durable result -- persist it to Drive, not the
    # local ephemeral WORK_DIR.
    out_path = os.path.join(INFERENCE_DIR, f"phase2_output_{video_basename}.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved: {out_path}")

    # Video-level risk rollup. Kept as a SEPARATE file from phase2_output_*
    # rather than wrapping the per-segment list in a dict, so eval_data()
    # above -- which reads phase2_output_*.json expecting a bare list of
    # segments -- doesn't need to change. There's no VIRAT ground truth for
    # "suspicious", so this isn't scored anywhere, only persisted/printed.
    #
    # video_risk_score is the MAX across segments, not an average: one
    # person doing something flaggable should flag the video regardless of
    # how many other unremarkable people/segments are in it.
    video_risk_score = max((r["risk_score"] for r in results), default=0)
    flagged_segments = [r["segment_id"] for r in results if r.get("suspicious")]
    risk_summary = {
        "video": video_basename,
        "num_segments": len(results),
        "video_suspicious": bool(flagged_segments),
        "video_risk_score": video_risk_score,
        "flagged_segments": flagged_segments,
    }
    risk_summary_path = os.path.join(INFERENCE_DIR, f"phase2_risk_{video_basename}.json")
    with open(risk_summary_path, "w") as f:
        json.dump(risk_summary, f, indent=2)
    print(
        f"Saved: {risk_summary_path}  "
        f"(video_risk_score={video_risk_score}, suspicious={risk_summary['video_suspicious']}, "
        f"flagged_segments={flagged_segments})"
    )

    total = t4 - t0
    def _pct(x):
        return f"{x:6.1f}s ({100 * x / total:4.1f}%)" if total > 0 else f"{x:6.1f}s"
    print(
        f"\n[timing] {video_file_name}\n"
        f"  [1/4] copy from Drive:   {_pct(t1 - t0)}\n"
        f"  [2/4] extract frames:    {_pct(t2 - t1)}\n"
        f"  [3/4] detect + track:    {_pct(t3 - t2)}\n"
        f"  [4/4] VLM describe:      {_pct(t4 - t3)}\n"
        f"  total:                   {total:6.1f}s"
    )

    return segments, results


def compute_prf1_accuracy(tp, fp, fn, tn=None):
    """Shared precision/recall/F1(/accuracy) computation.

    If `tn` is provided, accuracy is the standard (TP+TN)/(TP+TN+FP+FN).
    If `tn` is None (no well-defined negative set, e.g. bbox detection),
    a detection-style TP/(TP+FP+FN) proxy is returned instead.
    """
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    if tn is not None:
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
    else:
        accuracy = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    return round(precision, 4), round(recall, 4), round(f1, 4), round(accuracy, 4)


In [ ]:
"""
Phase 2 -- VIRAT-specific evaluation.

Depends on: the VLM-pipeline cell (CLASSES/chat_with_retry/TEXT_MODEL/
run_phase2) and the shared pipeline-driver cell (run_pipeline_for_video/
append_metrics_row/compute_prf1_accuracy/WORK_DIR/etc). Does NOT depend on
the UCF-Crime cell, and the UCF-Crime cell does not depend on this one --
either dataset can be run on its own.
"""

import json
from collections import defaultdict

import pandas as pd

# NOTE (redundancy removed): CLASSES was previously redefined here identical
# to the CLASSES list in the earlier VLM-pipeline cell. Both run in the same
# kernel, so this reuses that one instead of maintaining two copies that
# could silently drift out of sync with each other.


# Whether to LLM-judge each segment's free-text `summary` (from
# summarize_description) against (a) the chunk descriptions it was built
# from and (b) the VIRAT ground-truth activities in its time window. This
# is NOT a suspicious/risk judgment (that's assess_risk, a separate
# pipeline) -- purely: does the natural-language description match what
# actually happened. Costs one extra TEXT_MODEL call per segment during
# eval_data(); set False to skip it.
EVALUATE_SUMMARY_QUALITY = True

# ---------------------------------------------------------------------------
# PATHS
#
# Three roots, deliberately kept separate:
#
#   DATA_ROOT    (Drive) -- read-only source data: your existing VIRAT_DATA
#                            folder. Videos live directly inside it.
#   PROJECT_ROOT (Drive) -- durable outputs only, in their OWN folder next to
#                            (not inside) VIRAT_DATA: per-video inference
#                            JSON under inference/, plus the two result CSVs.
#                            This is the "where did my results go" folder.
#   WORK_DIR     (local Colab disk, NOT Drive) -- frames / boxed_frames /
#                            person_crops. These are large (thousands of
#                            JPEGs per video), fully ephemeral, and rewritten
#                            per video by clear_pipeline_dirs(). Writing that
#                            volume of small files to a Drive FUSE mount is
#                            the single biggest avoidable slowdown in this
#                            pipeline -- local disk is at minimum an order of
#                            magnitude faster for this access pattern, and
#                            nothing here needs to survive a runtime restart.
#
# ASSUMPTION: annotations live in VIRAT_DATA/annotations. Only videos were
# mentioned as being in this folder -- if the VIRAT annotation .txt files are
# somewhere else, change ANNOTATIONS_DIR below (nothing else depends on it
# except the evaluation step; comment that call out in main() if you don't
# have ground-truth annotations and just want tracking + descriptions).
# ---------------------------------------------------------------------------
DATA_ROOT = "/content/drive/MyDrive/VIRAT_DATA"
VIDEO_DIR = DATA_ROOT
ANNOTATIONS_DIR = os.path.join(DATA_ROOT, "annotations")

PROJECT_ROOT = "/content/drive/MyDrive/VIRAT_Results"
INFERENCE_DIR = os.path.join(PROJECT_ROOT, "inference")
CSV_PATH = os.path.join(PROJECT_ROOT, "eval-results.csv")
DATASET_SUMMARY_CSV_PATH = os.path.join(PROJECT_ROOT, "eval-results-dataset-summary.csv")

for d in (PROJECT_ROOT, INFERENCE_DIR):
    os.makedirs(d, exist_ok=True)

print("Reading videos from:      ", VIDEO_DIR)
print("Reading annotations from: ", ANNOTATIONS_DIR)
print("Saving results to:        ", PROJECT_ROOT)
print("  - per-video inference:  ", INFERENCE_DIR)
print("  - per-video metrics:    ", CSV_PATH)
print("  - dataset summary:      ", DATASET_SUMMARY_CSV_PATH)
print("Local scratch (not Drive):", WORK_DIR)


_print_gpu_status()


def compute_iou(box1, box2):
    """
    box format: [x1, y1, x2, y2]
    """
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter = max(0, xB - xA) * max(0, yB - yA)

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter

    if union == 0:
        return 0.0

    return inter / union


def _index_gt_by_frame(events_df: pd.DataFrame) -> dict:
    """frame number -> list of (df_index, [x1,y1,x2,y2], event_id, event_type).

    FIX: the original ran a full-DataFrame boolean scan plus .iterrows() for
    EVERY prediction -- O(detections x gt_rows). With tens of thousands of
    detections that dominated total runtime. Build the index once instead.
    """
    index = defaultdict(list)
    for row in events_df.itertuples(index=True):
        index[int(row.frame)].append((
            row.Index,
            [row.x, row.y, row.x2, row.y2],
            int(row.event_id),
            int(row.event_type),
        ))
    return index


def evaluate_matches(segments, events_df,
                     fps=FPS,
                     temporal_threshold=1,
                     iou_weight=0.8,
                     iou_threshold=0.5):
    """
    Evaluate predicted bounding boxes against VIRAT ground-truth boxes.

    `segments` is every tracked person's segment (from process_segments) --
    there is no reference-photo filtering upstream of this, so this scores
    detections against every person the tracker followed, not one queried
    individual.

    For every predicted detection, find the best-matching GT box within
    `temporal_threshold` frames (ranked by a blend of IoU + temporal
    closeness). Assignment is then ONE-TO-ONE (COCO/VOC convention): the
    highest-IoU prediction claims a GT box, and any further prediction
    matching that same box is a duplicate-detection False Positive. A
    prediction with no nearby GT candidate at all is also a False Positive.

    FALSE NEGATIVES: counted only over GT boxes that were actually
    EVALUABLE -- i.e. GT rows lying within `temporal_threshold` frames of
    some sampled/predicted frame. The original used len(events_df), which
    charged the model for ground-truth boxes on frames it was never shown,
    inflating FN and deflating recall.

    Returns
    -------
    results : list
        Per-prediction match records (includes an "is_tp" flag)
    metrics : dict
        Aggregate bbox metrics
    """
    if "x2" not in events_df.columns:
        events_df = events_df.copy()
        events_df["x2"] = events_df["x"] + events_df["w"]
        events_df["y2"] = events_df["y"] + events_df["h"]

    gt_index = _index_gt_by_frame(events_df)

    results = []
    ious = []
    temporal_errors = []

    matched_gt_indices = set()
    predicted_frames = set()
    candidate_pairs = []          # (record, gt_df_index) awaiting assignment
    unmatched_predictions = 0     # predictions with no GT in the time window
    tp = 0
    fp = 0

    for segment in segments:
        for det in segment["trajectory"]:

            pred_frame = round(det["timestamp"] * fps)
            pred_box = det["bbox"]
            predicted_frames.add(pred_frame)

            best = None
            best_score = -1.0
            best_idx = None

            for f in range(pred_frame - temporal_threshold, pred_frame + temporal_threshold + 1):
                for gt_idx, gt_box, event_id, event_type in gt_index.get(f, ()):

                    iou = compute_iou(pred_box, gt_box)
                    dt = abs(f - pred_frame)
                    temporal_similarity = (
                        1 - dt / temporal_threshold if temporal_threshold > 0 else 1
                    )
                    score = iou_weight * iou + (1 - iou_weight) * temporal_similarity

                    if score > best_score:
                        best_score = score
                        best_idx = gt_idx
                        best = {
                            "segment_id": segment["segment_id"],
                            "track_id": segment["track_id"],
                            "predicted_frame": pred_frame,
                            "ground_truth_frame": f,
                            "temporal_difference": dt,
                            "iou": round(iou, 4),
                            "score": round(score, 4),
                            "event_id": event_id,
                            # event_type is 1-indexed (1-12) in VIRAT
                            "event_type": event_type,
                            "event_name": (
                                CLASSES[event_type - 1]
                                if 1 <= event_type <= len(CLASSES) else "unknown"
                            ),
                        }

            if best is not None:
                results.append(best)
                candidate_pairs.append((best, best_idx))
                ious.append(best["iou"])
                temporal_errors.append(best["temporal_difference"])
            else:
                # No GT candidate at all within the temporal window -> FP
                unmatched_predictions += 1

    # FIX: one-to-one assignment. The original let EVERY prediction that
    # overlapped the same GT box count as a separate TP, so N duplicate boxes
    # on one person scored N true positives against a single annotation --
    # inflating both precision and recall. COCO/VOC resolve this greedily:
    # best-IoU prediction claims the GT box, every later prediction matching
    # the same box is a duplicate-detection FP.
    for best, gt_idx in sorted(candidate_pairs, key=lambda p: -p[0]["iou"]):
        if best["iou"] >= iou_threshold and gt_idx not in matched_gt_indices:
            best["is_tp"] = True
            matched_gt_indices.add(gt_idx)
            tp += 1
        else:
            best["is_tp"] = False
            best["fp_reason"] = (
                "duplicate_detection"
                if best["iou"] >= iou_threshold else "low_iou"
            )
            fp += 1

    fp += unmatched_predictions

    # Only GT boxes near a frame we actually evaluated can be "missed".
    evaluable_gt = 0
    for f, rows in gt_index.items():
        if any((f + d) in predicted_frames
               for d in range(-temporal_threshold, temporal_threshold + 1)):
            evaluable_gt += len(rows)
    fn = max(0, evaluable_gt - len(matched_gt_indices))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    # NOTE: classic accuracy = (TP+TN)/(TP+TN+FP+FN) requires a countable set
    # of negatives. For bounding-box detection there's no fixed universe of
    # "boxes that weren't drawn" -> TN is undefined, so real accuracy can't be
    # computed here (this is why COCO/PASCAL VOC-style detection evaluation
    # reports precision/recall/F1, never accuracy). What we report instead is
    # the detection "accuracy" proxy, TP/(TP+FP+FN).
    detection_accuracy = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0

    metrics = {
        "matched_boxes": len(results),
        "evaluable_gt_boxes": evaluable_gt,
        "total_gt_boxes": len(events_df),
        "mean_iou": round(float(np.mean(ious)), 4) if ious else 0,
        "mean_temporal_difference": round(float(np.mean(temporal_errors)), 4) if temporal_errors else 0,
        "bbox_true_positives": tp,
        "bbox_false_positives": fp,
        "bbox_false_negatives": fn,
        "bbox_precision": round(precision, 4),
        "bbox_recall": round(recall, 4),
        "bbox_f1": round(f1, 4),
        "bbox_detection_accuracy": round(detection_accuracy, 4),
    }

    # ious/temporal_errors are returned so main() can pool them exactly
    # instead of re-weighting rounded per-video means.
    return results, metrics, ious, temporal_errors


def parse_events_file(event_file: str) -> pd.DataFrame:
    """Parse a VIRAT '<clip>.viratdata.events.txt' file into a DataFrame.

    Columns per the VIRAT spec:
    event_id, event_type, duration, start_frame, end_frame, frame, x, y, w, h
    """
    columns = [
        "event_id", "event_type", "duration",
        "start_frame", "end_frame", "frame",
        "x", "y", "w", "h",
    ]
    rows = []
    with open(event_file, "r") as f:
        for line in f:
            parts = line.split()
            if len(parts) != len(columns):
                continue  # skip malformed/blank lines
            try:
                rows.append([int(p) for p in parts])
            except ValueError:
                continue
    return pd.DataFrame(rows, columns=columns)


# ---------------------------------------------------------------------------
# Summary-quality evaluation (the free-text description half of Phase 2)
#
# eval_data()'s activity-level TP/FP/FN below already scores the `activity`
# label list exactly, against VIRAT ground truth. It says nothing about the
# `summary` string itself, which is the other Phase 2 deliverable and has
# no VIRAT-provided reference text to diff against. This is purely a
# description-matching check, not a risk/suspicion judgment (that lives in
# assess_risk, upstream, and is out of scope here):
#
#   - faithfulness: does the summary actually follow from the underlying
#     per-chunk VLM descriptions it was built from, or did the
#     summarization step (mistral, text-only) invent detail not present
#     upstream? Isolates error introduced by summarize_description() itself
#     from whatever the vision model got wrong -- there's no ground truth
#     for the latter, so it isn't claimed here.
#   - coverage: of the VIRAT activities actually annotated in this
#     segment's time window, how many does the prose describe (in its own
#     words -- exact label text isn't required), and which watch-list
#     activities does it describe that VIRAT does NOT support for this
#     window (a text-level false-claim check, distinct from the activity
#     list's own FP count, since a segment can get the `activity` field
#     right per the schema while the surrounding prose describes it
#     inaccurately, or vice versa).
# ---------------------------------------------------------------------------

SUMMARY_EVAL_PROMPT_TEMPLATE = (
    "You are grading an AI-generated video activity summary for accuracy "
    "against source material. This is a factual-matching check, not a "
    "judgment about whether the behavior is suspicious.\n\n"
    "Underlying observations (what a vision model saw, in chronological "
    "chunks):\n{chunk_descriptions}\n\n"
    "Generated summary (produced by summarizing the observations above):\n"
    "{summary}\n\n"
    "Ground-truth activities known to have occurred in this time window, "
    "from human-annotated labels (may be empty if none of the watch-list "
    "activities occurred): {gt_activities}\n\n"
    "Grade the generated summary on two independent axes:\n\n"
    "1. FAITHFULNESS: does the summary accurately reflect the underlying "
    "observations, without inventing details, actions, or objects the "
    "observations don't support? A summary that condenses/paraphrases the "
    "observations faithfully should score high; one that adds specifics "
    "not present in the observations should score low.\n\n"
    "2. COVERAGE: for each ground-truth activity listed above, does the "
    "summary describe it in its own words (exact wording is not required)? "
    "List any ground-truth activities the summary fails to mention "
    "(missed_activities), and any watch-list activities the summary "
    "describes that are NOT in the ground-truth list "
    "(hallucinated_activities).\n\n"
    "Return:\n"
    "- faithfulness_score: integer 0-100\n"
    "- coverage_score: integer 0-100 (100 if there are no ground-truth "
    "activities to cover)\n"
    "- missed_activities: array of strings\n"
    "- hallucinated_activities: array of strings\n"
    "- notes: one sentence explaining the scores"
)

SUMMARY_EVAL_SCHEMA = {
    "type": "object",
    "properties": {
        "faithfulness_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "coverage_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "missed_activities": {"type": "array", "items": {"type": "string"}},
        "hallucinated_activities": {"type": "array", "items": {"type": "string"}},
        "notes": {"type": "string"},
    },
    "required": [
        "faithfulness_score", "coverage_score",
        "missed_activities", "hallucinated_activities", "notes",
    ],
}


def evaluate_summary_quality(summary: str, chunk_descriptions: list[str],
                             gt_activities: set, model: str = TEXT_MODEL) -> dict:
    """LLM-judge one segment's summary for faithfulness + GT coverage.

    Purely a description-matching check (does the text match what
    happened) -- no suspicious/risk judgment is made here.

    Short-circuits (no model call) when there's no summary to grade -- an
    empty segment already shows up as a false negative in the activity-level
    metrics below; this just mirrors that instead of paying a call for it.
    """
    if not summary:
        return {
            "faithfulness_score": 0,
            "coverage_score": 100 if not gt_activities else 0,
            "missed_activities": sorted(gt_activities),
            "hallucinated_activities": [],
            "notes": "No summary was generated for this segment.",
        }

    response = chat_with_retry(
        model=model,
        messages=[
            {
                "role": "user",
                "content": SUMMARY_EVAL_PROMPT_TEMPLATE.format(
                    chunk_descriptions="\n".join(chunk_descriptions) if chunk_descriptions else "(none)",
                    summary=summary,
                    gt_activities=", ".join(sorted(gt_activities)) if gt_activities else "(none)",
                ),
            }
        ],
        response_format=SUMMARY_EVAL_SCHEMA,
    )
    return json.loads(response["message"]["content"])


def eval_data(video_file_name: str, segments, fps: int = FPS,
              temporal_threshold: int = 1, iou_weight: float = 0.8,
              iou_threshold: float = 0.5):
    """
    Compare phase2's predicted per-segment activities against VIRAT
    ground-truth events overlapping that segment's time window, AND evaluate
    the predicted bounding boxes against the same ground truth.

    `segments` covers every tracked person in the video -- there is no
    reference-photo filtering, so this evaluates the full population.

    If EVALUATE_SUMMARY_QUALITY is True, each segment's free-text `summary`
    is also LLM-judged (see evaluate_summary_quality) for faithfulness to
    its underlying chunk descriptions and coverage of the ground-truth
    activities in its time window -- a description-matching check, not a
    suspicious/risk judgment.

    Returns (per_segment_results, bbox_results, metrics, ious,
    temporal_errors, summary_faithfulness_scores, summary_coverage_scores).
    The last two are [] when EVALUATE_SUMMARY_QUALITY is False.
    """
    video_name = os.path.splitext(video_file_name)[0]
    event_file = os.path.join(ANNOTATIONS_DIR, f"{video_name}.viratdata.events.txt")
    inference_path = os.path.join(INFERENCE_DIR, f"phase2_output_{video_name}.json")

    events_df = parse_events_file(event_file)
    events_df["x2"] = events_df["x"] + events_df["w"]
    events_df["y2"] = events_df["y"] + events_df["h"]

    with open(inference_path, "r") as f:
        inference_data = json.load(f)

    # ---- Activity (segment-level) evaluation ----
    per_segment_results = []
    tp = fp = fn = tn = 0
    num_classes = len(CLASSES)
    summary_faithfulness_scores = []
    summary_coverage_scores = []

    for segment in inference_data:
        # start_time / end_time are SECONDS, so * fps gives frames.
        start_frame = round(segment["start_time"] * fps)
        end_frame = round(segment["end_time"] * fps)

        gt_rows = events_df[
            (events_df["frame"] >= start_frame) &
            (events_df["frame"] <= end_frame)
        ]

        # event_type is 1-indexed in VIRAT (1-12); CLASSES is 0-indexed
        gt_activities = {
            CLASSES[int(et) - 1]
            for et in gt_rows["event_type"].unique()
            if 1 <= int(et) <= len(CLASSES)
        }
        # CAUTION: Phase 2 no longer produces an "activities" field at all
        # (see the open-vocabulary rewrite of CHUNK_PROMPT/
        # SUMMARY_PROMPT_TEMPLATE in the VLM-pipeline cell) -- .get() with a
        # default means this silently returns an empty set for every
        # segment from here on, rather than raising. That makes
        # precision/recall/f1 below trivially degenerate (recall pinned to
        # 0 whenever gt_activities is non-empty) for any video processed
        # after that change, NOT a sign the pipeline regressed. This
        # activity-level VIRAT benchmark only remains meaningful for
        # phase2_output files generated before the open-vocabulary rewrite.
        pred_activities = set(segment.get("activities", []))

        seg_tp = len(gt_activities & pred_activities)
        seg_fp = len(pred_activities - gt_activities)
        seg_fn = len(gt_activities - pred_activities)
        seg_tn = num_classes - seg_tp - seg_fp - seg_fn
        tp += seg_tp
        fp += seg_fp
        fn += seg_fn
        tn += seg_tn

        seg_result = {
            "segment_id": segment["segment_id"],
            "track_id": segment["track_id"],
            "start_time": segment["start_time"],
            "end_time": segment["end_time"],
            "ground_truth_activities": sorted(gt_activities),
            "predicted_activities": sorted(pred_activities),
            "true_positives": seg_tp,
            "false_positives": seg_fp,
            "false_negatives": seg_fn,
            "true_negatives": seg_tn,
        }

        if EVALUATE_SUMMARY_QUALITY:
            judged = evaluate_summary_quality(
                segment.get("summary", ""),
                segment.get("descriptions", []),
                gt_activities,
            )
            seg_result["summary_faithfulness_score"] = judged["faithfulness_score"]
            seg_result["summary_coverage_score"] = judged["coverage_score"]
            seg_result["summary_missed_activities"] = judged["missed_activities"]
            seg_result["summary_hallucinated_activities"] = judged["hallucinated_activities"]
            seg_result["summary_eval_notes"] = judged["notes"]
            summary_faithfulness_scores.append(judged["faithfulness_score"])
            summary_coverage_scores.append(judged["coverage_score"])

        per_segment_results.append(seg_result)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    # NOTE: with 12 classes and typically 0-2 present per segment, seg_tn is
    # ~10 every time, so this accuracy sits near 0.85 no matter how the model
    # performs. Report it if you must, but precision/recall/F1 are the honest
    # numbers here.
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0

    # ---- Bounding-box evaluation ----
    bbox_results, bbox_metrics, ious, temporal_errors = evaluate_matches(
        segments,
        events_df,
        fps=fps,
        temporal_threshold=temporal_threshold,
        iou_weight=iou_weight,
        iou_threshold=iou_threshold,
    )

    metrics = {
        "video": video_name,
        "num_segments": len(inference_data),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4),
        **bbox_metrics,
        "mean_summary_faithfulness": (
            round(float(np.mean(summary_faithfulness_scores)), 4)
            if summary_faithfulness_scores else 0
        ),
        "mean_summary_coverage": (
            round(float(np.mean(summary_coverage_scores)), 4)
            if summary_coverage_scores else 0
        ),
    }

    append_metrics_row(CSV_PATH, metrics)

    return (per_segment_results, bbox_results, metrics, ious, temporal_errors,
            summary_faithfulness_scores, summary_coverage_scores)


def main():
    """Run the full pipeline + evaluation over every video in VIDEO_DIR.

    Every person the tracker follows in every video gets described and
    evaluated -- there is no reference-photo step to opt a subset in."""
    video_files = sorted(
        f for f in os.listdir(VIDEO_DIR)
        if f.lower().endswith((".mp4", ".avi", ".mov"))
    )

    ds_act_tp = ds_act_fp = ds_act_fn = ds_act_tn = 0
    ds_bbox_tp = ds_bbox_fp = ds_bbox_fn = 0
    ds_ious = []
    ds_temporal_errors = []
    ds_summary_faithfulness = []
    ds_summary_coverage = []
    ds_matched_boxes = 0
    ds_num_segments = 0
    ds_videos_evaluated = 0
    ds_failures = []

    for video_file_name in tqdm(video_files, desc="Processing videos"):
        try:
            segments, results = run_pipeline_for_video(video_file_name)
            (per_segment_results, bbox_results, metrics, ious, temporal_errors,
             summary_faithfulness_scores, summary_coverage_scores) = eval_data(
                video_file_name, segments
            )
            print(f"{video_file_name}: {metrics}")

            ds_act_tp += metrics["true_positives"]
            ds_act_fp += metrics["false_positives"]
            ds_act_fn += metrics["false_negatives"]
            ds_act_tn += metrics["true_negatives"]
            ds_bbox_tp += metrics["bbox_true_positives"]
            ds_bbox_fp += metrics["bbox_false_positives"]
            ds_bbox_fn += metrics["bbox_false_negatives"]
            ds_matched_boxes += metrics["matched_boxes"]
            ds_num_segments += metrics["num_segments"]
            ds_videos_evaluated += 1

            # FIX: pool the raw per-box values instead of replicating a
            # rounded per-video mean.
            ds_ious.extend(ious)
            ds_temporal_errors.extend(temporal_errors)
            ds_summary_faithfulness.extend(summary_faithfulness_scores)
            ds_summary_coverage.extend(summary_coverage_scores)

        except Exception as e:
            # FIX: the bare `print(e)` hid the real stack trace -- which is how
            # a TypeError inside run_phase2 turned into every video silently
            # "failing" with no explanation.
            ds_failures.append((video_file_name, repr(e)))
            print(f"Failed on {video_file_name}: {e}", file=sys.stderr)
            traceback.print_exc()

    act_precision, act_recall, act_f1, act_accuracy = compute_prf1_accuracy(
        ds_act_tp, ds_act_fp, ds_act_fn, tn=ds_act_tn
    )
    bbox_precision, bbox_recall, bbox_f1, bbox_detection_accuracy = compute_prf1_accuracy(
        ds_bbox_tp, ds_bbox_fp, ds_bbox_fn, tn=None
    )

    dataset_metrics = {
        "videos_total": len(video_files),
        "videos_evaluated": ds_videos_evaluated,
        "videos_failed": len(ds_failures),
        "num_segments": ds_num_segments,
        "true_positives": ds_act_tp,
        "false_positives": ds_act_fp,
        "false_negatives": ds_act_fn,
        "true_negatives": ds_act_tn,
        "precision": act_precision,
        "recall": act_recall,
        "f1": act_f1,
        "accuracy": act_accuracy,
        "matched_boxes": ds_matched_boxes,
        "mean_iou": round(float(np.mean(ds_ious)), 4) if ds_ious else 0,
        "mean_temporal_difference": round(float(np.mean(ds_temporal_errors)), 4) if ds_temporal_errors else 0,
        "mean_summary_faithfulness": round(float(np.mean(ds_summary_faithfulness)), 4) if ds_summary_faithfulness else 0,
        "mean_summary_coverage": round(float(np.mean(ds_summary_coverage)), 4) if ds_summary_coverage else 0,
        "bbox_true_positives": ds_bbox_tp,
        "bbox_false_positives": ds_bbox_fp,
        "bbox_false_negatives": ds_bbox_fn,
        "bbox_precision": bbox_precision,
        "bbox_recall": bbox_recall,
        "bbox_f1": bbox_f1,
        "bbox_detection_accuracy": bbox_detection_accuracy,
    }

    with open(DATASET_SUMMARY_CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(dataset_metrics.keys()))
        writer.writeheader()
        writer.writerow(dataset_metrics)

    print(f"\nDataset-wide summary ({ds_videos_evaluated} videos): {dataset_metrics}")
    if ds_failures:
        print(f"\n{len(ds_failures)} video(s) failed:")
        for name, err in ds_failures:
            print(f"  - {name}: {err}")
    print(f"Saved: {DATASET_SUMMARY_CSV_PATH}")


if __name__ == "__main__":
    main()


In [ ]:
"""
Phase 2 -- UCF-Crime shared utilities: paths, manifest loading, fps
detection, caption alignment, and the per-video scoring function.

Separated from the full-dataset driver (next cell) so a spot-check or
diagnostic on a handful of videos doesn't require running the full
53-video loop -- this cell only DEFINES functions, it doesn't run anything.

Depends on: the VLM-pipeline cell (chat_with_retry/TEXT_MODEL/run_phase2)
and the shared pipeline-driver cell (run_pipeline_for_video/
append_metrics_row/compute_prf1_accuracy/WORK_DIR/etc). Does NOT depend on
the VIRAT-specific eval cell -- this cell can run entirely on its own, right
after the VLM-pipeline and shared-driver cells, without the VIRAT cell ever
executing.

Conceptually parallel to the VIRAT eval_data(), but for the UCF-Crime
subset produced by prepare_ucf_crime_subset.py (a folder of .mp4 files plus
one subset_manifest.json: {video: {category, anomaly_frame_windows,
captions}}). UCF-Crime has no bounding-box ground truth and no per-activity
labels -- what it DOES give us, that VIRAT couldn't, is real ground truth
for exactly the two things that had none before:

  (a) video-level suspicious/not-suspicious classification: UCF-Crime's
      `category` (Anomaly type, or "Normal") is real ground truth for the
      assess_risk-derived `video_suspicious` rollup written by
      run_pipeline_for_video. Scored as a binary classifier: precision,
      recall, F1, accuracy, pooled across videos (reusing
      compute_prf1_accuracy from the VIRAT cell above).

  (b) natural-language description matching: UCA's captions (sentences +
      per-second timestamps) are real human-written reference text for
      each segment's `summary`, aligned by time overlap. Scored on the same
      faithfulness-vs-chunk-descriptions axis as evaluate_summary_quality
      (VIRAT cell), plus a new agreement-vs-reference-text axis -- there is
      no fixed activity list to check coverage against here, by design (see
      the open-vocabulary rewrite in the VLM-pipeline cell).

Reuses run_pipeline_for_video (shared cell) / run_phase2 / assess_risk
(VLM-pipeline cell) unchanged -- this cell only adds the UCF-Crime-specific
ground truth + scoring on top.
"""

# ---------------------------------------------------------------------------
# PATHS -- separate from the VIRAT ones above so nothing here overwrites
# VIRAT results. run_pipeline_for_video() reads VIDEO_DIR/INFERENCE_DIR as
# module-level globals (see the cell above), so this reassigns those globals
# before calling it rather than duplicating ~80 lines of pipeline code.
# ---------------------------------------------------------------------------
UCF_VIDEO_DIR = "/content/drive/MyDrive/ucf_crime_subset"      # output of prepare_ucf_crime_subset.py --out
UCF_MANIFEST_PATH = os.path.join(UCF_VIDEO_DIR, "subset_manifest.json")

UCF_PROJECT_ROOT = "/content/drive/MyDrive/UCF_Crime_Results"
UCF_INFERENCE_DIR = os.path.join(UCF_PROJECT_ROOT, "inference")
UCF_CSV_PATH = os.path.join(UCF_PROJECT_ROOT, "eval-results.csv")
UCF_DATASET_SUMMARY_CSV_PATH = os.path.join(UCF_PROJECT_ROOT, "eval-results-dataset-summary.csv")

for d in (UCF_PROJECT_ROOT, UCF_INFERENCE_DIR):
    os.makedirs(d, exist_ok=True)


def load_manifest(manifest_path: str) -> dict:
    with open(manifest_path) as f:
        manifest = json.load(f)
    n_with_captions = sum(1 for v in manifest.values() if v.get("captions"))
    print(f"[load_manifest] {len(manifest)} videos "
          f"({n_with_captions} with UCA captions)")
    return manifest


def get_video_fps(video_path: str) -> float:
    """ffprobe the video's real frame rate rather than assuming a constant.

    This is deliberately NOT hardcoded to FPS=30 -- the exact bug class
    that corrupted the VIRAT timestamps earlier in this notebook (30 vs
    29.97) is exactly the kind of thing a "should be 30" assumption misses.
    Falls back to 30.0 only if ffprobe itself fails.
    """
    try:
        out = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=avg_frame_rate", "-of", "csv=p=0", video_path],
            capture_output=True, text=True, timeout=15,
        )
        raw = out.stdout.strip()
        if "/" in raw:
            num, den = raw.split("/")
            den = float(den)
            return float(num) / den if den else float(num)
        return float(raw) if raw else 30.0
    except Exception as e:
        print(f"[get_video_fps] WARNING: could not probe {video_path} ({e}); assuming 30.0")
        return 30.0


# ---------------------------------------------------------------------------
# (b) Description matching against UCA reference captions
# ---------------------------------------------------------------------------

REFERENCE_EVAL_PROMPT_TEMPLATE = (
    "You are grading an AI-generated video activity summary against a "
    "human-written reference description of the same time window. This is "
    "a factual-matching check, not a judgment about whether the behavior "
    "is suspicious.\n\n"
    "Underlying observations (what a vision model saw, in chronological "
    "chunks):\n{chunk_descriptions}\n\n"
    "Generated summary (produced by summarizing the observations above):\n"
    "{summary}\n\n"
    "Human-written reference description(s) of this same time window:\n"
    "{reference_text}\n\n"
    "Grade the generated summary on two independent axes:\n\n"
    "1. FAITHFULNESS: does the summary accurately reflect the underlying "
    "observations, without inventing details the observations don't "
    "support?\n\n"
    "2. AGREEMENT: does the summary describe the same events as the "
    "reference description(s), without missing or contradicting anything "
    "material in them? List any events the reference mentions that the "
    "summary misses (missed_events), and anything the summary claims that "
    "the reference does not support (unsupported_claims).\n\n"
    "Return:\n"
    "- faithfulness_score: integer 0-100\n"
    "- agreement_score: integer 0-100\n"
    "- missed_events: array of strings\n"
    "- unsupported_claims: array of strings\n"
    "- notes: one sentence explaining the scores"
)

REFERENCE_EVAL_SCHEMA = {
    "type": "object",
    "properties": {
        "faithfulness_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "agreement_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "missed_events": {"type": "array", "items": {"type": "string"}},
        "unsupported_claims": {"type": "array", "items": {"type": "string"}},
        "notes": {"type": "string"},
    },
    "required": [
        "faithfulness_score", "agreement_score",
        "missed_events", "unsupported_claims", "notes",
    ],
}


def find_overlapping_captions(seg_start: float, seg_end: float, captions_entry) -> list[str]:
    """UCA sentences whose timestamp span overlaps [seg_start, seg_end] at
    all. Returns [] (not evaluable) if there's no captions entry or no
    overlapping sentence -- callers should skip scoring in that case rather
    than treating it as a zero, same convention as VIRAT's evaluable_gt."""
    if not captions_entry:
        return []
    overlapping = []
    for sentence, (t0, t1) in zip(captions_entry["sentences"], captions_entry["timestamps"]):
        if min(seg_end, t1) - max(seg_start, t0) > 0:
            overlapping.append(sentence)
    return overlapping


def evaluate_summary_against_reference(summary: str, chunk_descriptions: list[str],
                                       reference_text: str, model: str = TEXT_MODEL) -> dict:
    """LLM-judge one segment's summary against real UCA reference text.

    Same faithfulness axis as evaluate_summary_quality; agreement replaces
    coverage since there's no fixed activity vocabulary to check against
    here, only free text.
    """
    if not summary or not reference_text:
        return {
            "faithfulness_score": 0,
            "agreement_score": 0,
            "missed_events": [],
            "unsupported_claims": [],
            "notes": "Missing summary or reference text for this segment.",
        }

    response = chat_with_retry(
        model=model,
        messages=[
            {
                "role": "user",
                "content": REFERENCE_EVAL_PROMPT_TEMPLATE.format(
                    chunk_descriptions="\n".join(chunk_descriptions) if chunk_descriptions else "(none)",
                    summary=summary,
                    reference_text=reference_text,
                ),
            }
        ],
        response_format=REFERENCE_EVAL_SCHEMA,
    )
    return json.loads(response["message"]["content"])


# ---------------------------------------------------------------------------
# (a) + (b) combined per-video evaluation
# ---------------------------------------------------------------------------

def eval_video_ucf_crime(video_file_name: str, phase2_results: list, risk_summary: dict,
                         manifest_entry: dict) -> dict:
    """Score one video's suspicious classification (a) and, for every
    segment with an overlapping UCA caption, its description quality (b).

    Returns a metrics dict (appended to UCF_CSV_PATH) plus the raw
    per-segment faithfulness/agreement scores for dataset-wide pooling in
    run_ucf_crime_pipeline (same "pool raw values, don't average rounded
    per-video means" convention as the VIRAT cell).
    """
    video_name = os.path.splitext(video_file_name)[0]
    category = manifest_entry["category"]
    captions_entry = manifest_entry.get("captions")

    # ---- (a) video-level suspicious classification ----
    gt_suspicious = category != "Normal"
    pred_suspicious = bool(risk_summary["video_suspicious"])

    # ---- (b) per-segment description matching against UCA captions ----
    faithfulness_scores = []
    agreement_scores = []
    seg_results = []

    for segment in phase2_results:
        refs = find_overlapping_captions(segment["start_time"], segment["end_time"], captions_entry)
        if not refs:
            continue  # no reference text for this window -- not evaluable

        judged = evaluate_summary_against_reference(
            segment.get("summary", ""),
            segment.get("descriptions", []),
            " ".join(refs),
        )
        faithfulness_scores.append(judged["faithfulness_score"])
        agreement_scores.append(judged["agreement_score"])
        seg_results.append({
            "segment_id": segment["segment_id"],
            "track_id": segment["track_id"],
            "start_time": segment["start_time"],
            "end_time": segment["end_time"],
            "reference_text": " ".join(refs),
            "faithfulness_score": judged["faithfulness_score"],
            "agreement_score": judged["agreement_score"],
            "missed_events": judged["missed_events"],
            "unsupported_claims": judged["unsupported_claims"],
        })

    metrics = {
        "video": video_name,
        "category": category,
        "gt_suspicious": gt_suspicious,
        "pred_suspicious": pred_suspicious,
        "correct_suspicious_classification": gt_suspicious == pred_suspicious,
        "video_risk_score": risk_summary["video_risk_score"],
        "num_segments": len(phase2_results),
        "num_segments_with_reference": len(seg_results),
        "mean_summary_faithfulness": (
            round(float(np.mean(faithfulness_scores)), 4) if faithfulness_scores else 0
        ),
        "mean_summary_agreement": (
            round(float(np.mean(agreement_scores)), 4) if agreement_scores else 0
        ),
    }

    append_metrics_row(UCF_CSV_PATH, metrics)

    return seg_results, metrics, faithfulness_scores, agreement_scores


In [ ]:
"""
Phase 2 -- UCF-Crime full-dataset run.

Iterates every video in the UCF-Crime subset manifest through the shared
pipeline driver and the scoring function from the previous cell, pools
dataset-wide metrics, and writes UCF_DATASET_SUMMARY_CSV_PATH. This is the
expensive, all-53-videos entry point -- for a quick check on one or two
videos, use the spot-check cell instead (it doesn't need this cell at all).

Depends on: the UCF-Crime shared-utilities cell above (load_manifest/
get_video_fps/eval_video_ucf_crime/UCF_* paths), the VLM-pipeline cell, and
the shared pipeline-driver cell. Running this cell executes the full
pipeline immediately (see the `if __name__ == "__main__"` block at the
bottom) -- don't run it just to get the function definitions.
"""


def run_ucf_crime_pipeline():
    """Run the pipeline + (a)/(b) evaluation over every video in the
    UCF-Crime subset manifest."""
    manifest = load_manifest(UCF_MANIFEST_PATH)
    video_files = sorted(
        v for v in manifest
        if os.path.exists(os.path.join(UCF_VIDEO_DIR, v))
    )
    missing = set(manifest) - set(video_files)
    if missing:
        print(f"[run_ucf_crime_pipeline] {len(missing)} video(s) in the manifest "
              f"are not present in {UCF_VIDEO_DIR}, skipping: {sorted(missing)[:5]}...")

    # Point run_pipeline_for_video's globals at the UCF-Crime paths instead
    # of the VIRAT ones set up in the cell above.
    global VIDEO_DIR, INFERENCE_DIR
    VIDEO_DIR = UCF_VIDEO_DIR
    INFERENCE_DIR = UCF_INFERENCE_DIR

    ds_tp = ds_fp = ds_fn = ds_tn = 0
    ds_faithfulness = []
    ds_agreement = []
    ds_videos_evaluated = 0
    ds_failures = []

    for video_file_name in tqdm(video_files, desc="Processing UCF-Crime videos"):
        try:
            fps = get_video_fps(os.path.join(UCF_VIDEO_DIR, video_file_name))
            print(f"[run_ucf_crime_pipeline] {video_file_name}: detected fps={fps:.3f}")

            segments, phase2_results = run_pipeline_for_video(video_file_name, fps=round(fps, 3))

            video_name = os.path.splitext(video_file_name)[0]
            risk_path = os.path.join(UCF_INFERENCE_DIR, f"phase2_risk_{video_name}.json")
            with open(risk_path) as f:
                risk_summary = json.load(f)

            seg_results, metrics, faithfulness_scores, agreement_scores = eval_video_ucf_crime(
                video_file_name, phase2_results, risk_summary, manifest[video_file_name]
            )
            print(f"{video_file_name}: {metrics}")

            ds_tp += metrics["gt_suspicious"] and metrics["pred_suspicious"]
            ds_fp += (not metrics["gt_suspicious"]) and metrics["pred_suspicious"]
            ds_fn += metrics["gt_suspicious"] and (not metrics["pred_suspicious"])
            ds_tn += (not metrics["gt_suspicious"]) and (not metrics["pred_suspicious"])
            ds_faithfulness.extend(faithfulness_scores)
            ds_agreement.extend(agreement_scores)
            ds_videos_evaluated += 1

        except Exception as e:
            ds_failures.append((video_file_name, repr(e)))
            print(f"Failed on {video_file_name}: {e}", file=sys.stderr)
            traceback.print_exc()

    precision, recall, f1, accuracy = compute_prf1_accuracy(ds_tp, ds_fp, ds_fn, tn=ds_tn)

    dataset_metrics = {
        "videos_total": len(video_files),
        "videos_evaluated": ds_videos_evaluated,
        "videos_failed": len(ds_failures),
        "suspicious_true_positives": ds_tp,
        "suspicious_false_positives": ds_fp,
        "suspicious_false_negatives": ds_fn,
        "suspicious_true_negatives": ds_tn,
        "suspicious_precision": precision,
        "suspicious_recall": recall,
        "suspicious_f1": f1,
        "suspicious_accuracy": accuracy,
        "num_segments_scored_for_description": len(ds_faithfulness),
        "mean_summary_faithfulness": round(float(np.mean(ds_faithfulness)), 4) if ds_faithfulness else 0,
        "mean_summary_agreement": round(float(np.mean(ds_agreement)), 4) if ds_agreement else 0,
    }

    with open(UCF_DATASET_SUMMARY_CSV_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(dataset_metrics.keys()))
        writer.writeheader()
        writer.writerow(dataset_metrics)

    print(f"\nUCF-Crime dataset-wide summary ({ds_videos_evaluated} videos): {dataset_metrics}")
    if ds_failures:
        print(f"\n{len(ds_failures)} video(s) failed:")
        for name, err in ds_failures:
            print(f"  - {name}: {err}")
    print(f"Saved: {UCF_DATASET_SUMMARY_CSV_PATH}")

    return dataset_metrics


if __name__ == "__main__":
    run_ucf_crime_pipeline()


In [ ]:
"""
Phase 2 -- spot-check a small set of UCF-Crime videos after the
CHUNK_PROMPT/SUMMARY_PROMPT_TEMPLATE update (added Aggressive/Fighting/
Distress/Abandoning Object categories + explicit person-to-person
interaction wording), before committing to a full 53-video re-run.

Backs up each video's existing phase2_output_*.json / phase2_risk_*.json
(if present) to a "*_before_prompt_update.json" copy first --
run_pipeline_for_video() overwrites those files in place, so without this
the "before" result would be gone the moment the new run starts. Old
per-video metrics are read from UCF_CSV_PATH before the rerun for the same
reason (append_metrics_row only appends, so the old row still exists in the
CSV after this runs too -- if you re-run the same video repeatedly, filter
to the LAST matching row, which _read_existing_metrics already does).

Depends on: the VLM-pipeline cell, the shared pipeline-driver cell, and the
UCF-Crime SHARED-UTILITIES cell (load_manifest / eval_video_ucf_crime /
UCF_* paths / get_video_fps) -- deliberately NOT the UCF-Crime full-dataset
run cell, so a spot-check never requires running (or even defining) the
expensive all-videos loop. Run this after those three, not standalone.
"""

import csv
import shutil


def _read_existing_metrics(video_name: str):
    """Most recent UCF_CSV_PATH row for this video, or None if there isn't one."""
    if not os.path.exists(UCF_CSV_PATH):
        return None
    with open(UCF_CSV_PATH, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["video"] == video_name]
    return rows[-1] if rows else None


def _backup_existing_outputs(video_name: str) -> None:
    for prefix in ("phase2_output_", "phase2_risk_"):
        src = os.path.join(UCF_INFERENCE_DIR, f"{prefix}{video_name}.json")
        if os.path.exists(src):
            dst = os.path.join(UCF_INFERENCE_DIR, f"{prefix}{video_name}_before_prompt_update.json")
            shutil.copyfile(src, dst)
            print(f"[spot_check] backed up {src} -> {dst}")


_COMPARE_KEYS = ("pred_suspicious", "video_risk_score",
                "mean_summary_faithfulness", "mean_summary_agreement")


def spot_check_videos(video_files: list[str]) -> None:
    """Re-run just the given videos (filenames, with extension) and print
    old-vs-new metrics side by side."""
    manifest = load_manifest(UCF_MANIFEST_PATH)

    global VIDEO_DIR, INFERENCE_DIR
    VIDEO_DIR = UCF_VIDEO_DIR
    INFERENCE_DIR = UCF_INFERENCE_DIR

    for video_file_name in video_files:
        video_name = os.path.splitext(video_file_name)[0]

        if video_file_name not in manifest:
            print(f"[spot_check] WARNING: {video_file_name} not in manifest, skipping")
            continue

        old_metrics = _read_existing_metrics(video_name)
        _backup_existing_outputs(video_name)

        fps = get_video_fps(os.path.join(UCF_VIDEO_DIR, video_file_name))
        segments, phase2_results = run_pipeline_for_video(video_file_name, fps=round(fps, 3))

        risk_path = os.path.join(UCF_INFERENCE_DIR, f"phase2_risk_{video_name}.json")
        with open(risk_path) as f:
            risk_summary = json.load(f)

        _, new_metrics, _, _ = eval_video_ucf_crime(
            video_file_name, phase2_results, risk_summary, manifest[video_file_name]
        )

        print(f"\n=== {video_file_name} ({manifest[video_file_name]['category']}) ===")
        if old_metrics:
            print("  BEFORE:", {k: old_metrics[k] for k in _COMPARE_KEYS})
        else:
            print("  BEFORE: no prior row found in", UCF_CSV_PATH)
        print("  AFTER: ", {k: new_metrics[k] for k in _COMPARE_KEYS})


if __name__ == "__main__":
    spot_check_videos(["Abuse030_x264.mp4", "Arrest001_x264.mp4"])


In [ ]:
"""
Phase 2 -- diagnostic: print each segment's generated summary + raw VLM
chunk descriptions next to its matched UCA reference caption(s), for one
already-processed video.

Purpose: the aggregate faithfulness/agreement scores can't tell apart two
very different failure modes -- a vision-perception failure (the VLM's
chunk descriptions talk about something unrelated to what actually
happened) versus a borderline-scoring one (close, but the judge marked it
down for a specific missing/extra detail). Reading the actual text side by
side settles which one it is. Deliberately makes zero model calls -- it
only reads the phase2_output_<video>.json already on disk plus the
manifest, so it's free and instant to re-run.

Depends on: the UCF-Crime SHARED-UTILITIES cell (load_manifest /
find_overlapping_captions / UCF_MANIFEST_PATH / UCF_INFERENCE_DIR) --
not the full-dataset run cell. Reads existing output rather than running
the pipeline -- run this after a video has already been processed (by
run_ucf_crime_pipeline or spot_check_videos).
"""


def print_segment_diagnostics(video_file_name: str) -> None:
    video_name = os.path.splitext(video_file_name)[0]

    manifest = load_manifest(UCF_MANIFEST_PATH)
    if video_file_name not in manifest:
        print(f"[print_segment_diagnostics] {video_file_name} not in manifest")
        return
    captions_entry = manifest[video_file_name].get("captions")

    output_path = os.path.join(UCF_INFERENCE_DIR, f"phase2_output_{video_name}.json")
    if not os.path.exists(output_path):
        print(f"[print_segment_diagnostics] {output_path} not found -- "
              f"has this video been processed yet?")
        return
    with open(output_path) as f:
        phase2_results = json.load(f)

    print(f"=== {video_file_name} ({manifest[video_file_name]['category']}) -- "
          f"{len(phase2_results)} segment(s) ===\n")

    for segment in phase2_results:
        refs = find_overlapping_captions(segment["start_time"], segment["end_time"], captions_entry)

        print(f"--- segment {segment['segment_id']} (track {segment['track_id']}), "
              f"{segment['start_time']:.1f}s-{segment['end_time']:.1f}s ---")

        print("GENERATED SUMMARY:")
        print(f"  {segment.get('summary', '(none)')}")

        print("REFERENCE CAPTION(S):")
        if refs:
            for r in refs:
                print(f"  - {r}")
        else:
            print("  (none overlapping this window)")

        descriptions = segment.get("descriptions", [])
        if descriptions:
            print("RAW CHUNK DESCRIPTIONS (per VLM call, chronological):")
            for i, d in enumerate(descriptions):
                snippet = d if len(d) <= 400 else d[:400] + "..."
                print(f"  [chunk {i}] {snippet}")

        print("=" * 70)
        print()


if __name__ == "__main__":
    print_segment_diagnostics("Arrest001_x264.mp4")


# Known limitation: multi-person forced/coercive interactions in low-resolution, wide-shot footage

While validating Phase 2 on UCF-Crime, one video (`Arrest001_x264.mp4`) surfaced a
failure mode worth documenting rather than fully solving, given the architectural
cost of a real fix and the time remaining in this project.

## The finding

A human-confirmed physical altercation (a man pushed from behind, surrounded, and
dragged by two others) was consistently missed across every one of the 4 tracked
segments covering that time window -- not just once, but across three different,
independent attempted fixes:

1. **Expanding the activity checklist** (`CLASSES`) with DIVA-grounded
   violence/distress categories (Aggressive, Fighting, Distress, Abandoning
   Object) -- ruled out a missing-vocabulary explanation. The model still
   didn't describe the incident even once a word for it existed in the prompt.
2. **Cropping frames to each track's own (padded) bounding box**
   (`VLM_USE_CROPS`) -- this genuinely helped: raw chunk descriptions went
   from reporting no interaction at all to describing real person-to-person
   contact ("handling an object together", "approaches and gestures"). But the
   model consistently read forced contact as *cooperative* contact -- it saw
   the interaction, and got its nature backwards.
3. **Explicit force/motion prompting** (asking the model to compare posture
   across frames for signs of coercion vs. voluntary movement) -- had no
   measurable effect on the vision stage's actual output. The raw per-chunk
   descriptions were structurally unchanged; only the downstream text
   summarizer's phrasing changed (explicitly writing negations like "no
   evidence of aggression"), which is a symptom of the summarizer reasoning
   over an unchanged, still-mundane set of descriptions, not evidence the
   vision model perceived anything new.

## Why this is being documented, not chased further

Two converging, evidence-backed root causes, neither of which a fourth prompt
iteration is likely to fix:

- **No motion signal.** The pipeline samples ~1 frame/second and describes
  static images. Distinguishing "being dragged" from "walking alongside
  someone" fundamentally requires force/velocity information that isn't
  present in an isolated still frame, regardless of how the model is asked to
  look at it.
- **Per-track description can't represent inherently relational events.**
  `run_phase2` describes one tracked individual's segment at a time. A
  coordinated multi-person assault is a relationship between several
  people -- describing each participant's crop in isolation may structurally
  lose the "surrounded by two people" framing even when each individual crop
  is well-resolved. Fixing this would require joint multi-track framing (show
  everyone active in a time window in one VLM call), which changes the core
  segment data model (`segment_id`/`track_id` semantics, the UCF-Crime
  alignment code, and possibly VIRAT's bbox eval) -- a materially bigger
  change than anything else in this notebook, for a benefit that isn't
  guaranteed given the motion-signal problem would still remain.

**Conclusion:** this pipeline reliably describes activities that are visible
within a single, sufficiently-resolved crop and don't depend on distinguishing
force from cooperation. It is less reliable on events whose defining feature
*is* that distinction (coercion, restraint, non-consensual movement) when the
only evidence is a handful of low-resolution, temporally sparse still frames.
This is reported as a known limitation of the frame/track-based VLM
description architecture rather than a solved problem.

## Separately: a real bug found along the way, and fixed

Investigating this surfaced an unrelated but genuine correctness issue in
`summarize_description()`: `SUMMARY_PROMPT_TEMPLATE` instructed the model to
respond in a free-text `"Summary: ...\nDetected Activities:\n- ..."` bullet
format, while the actual call constrains output via
`response_format=SUMMARY_FORMAT_SCHEMA` to a `{"activity": [...], "summary":
str}` JSON object -- the model's own instructions and what it was actually
forced to emit directly contradicted each other. This is a plausible
explanation for a segment observed listing all four new danger categories as
"detected" while its own summary text said the opposite, with nothing in the
underlying observations supporting any of them -- not a content
hallucination, a structural one. The prompt has been rewritten to describe
the actual JSON output directly (matching the style already used by
`RISK_PROMPT_TEMPLATE`/`SUMMARY_EVAL_PROMPT_TEMPLATE`/
`REFERENCE_EVAL_PROMPT_TEMPLATE`, which never had this mismatch), with an
explicit instruction not to include an activity without clear supporting
evidence. This is a general reliability fix, independent of the limitation
above, and worth re-running the full evaluation to see if it reduces
activity-list noise dataset-wide.


---

# Known limitation: motion blur fragments detection during the exact moments a surveillance pipeline most needs to catch

While validating the pipeline against `Abuse030_x264.mp4` (whose reference
caption confirms a man beating a dog with a belt at ~42-45s), a 7.2-second
gap turned up between two tracked segments (`37.2-40.3s` and `47.5-48.6s`)
-- squarely swallowing the abuse itself. No VLM call, with any model
(confirmed on both `qwen2.5vl:7b` and `qwen3-vl:8b`), ever saw that
footage, because Phase 1's tracker never produced a trajectory entry for
any frame in that window.

## What `diagnose_detection_confidence` found

Running YOLO directly (no confidence floor) across 39.0-48.0s, frame by
frame:

- **0 frames with zero person-candidates at all** -- the person was never
  literally invisible to the detector.
- **253 frames with a candidate below the 0.5 threshold** -- but not one
  clean weak box per frame. Frame counts of *simultaneous* candidates
  ranged from 5 to 39 in a single frame, with confidence clustered at
  0.05-0.25 throughout the entire blurred window. This is detection
  FRAGMENTATION, not a single box narrowly missing the bar.
- **18 frames above 0.5** -- all landing at 47.433s onward, exactly where
  segment 4 already begins. Detection recovers cleanly and instantly
  (`n_candidates` drops from double digits to exactly 1; confidence jumps
  straight to 0.48 -> 0.55 -> 0.65 -> 0.79 -> 0.81) the moment the violent
  motion ends. This rules out a hidden tracker-continuity bug (case (c)
  from the diagnostic's own framing) -- the tracker correctly re-acquires
  the person as soon as detection recovers; there's nothing for it to have
  missed.

## Why this isn't a tunable parameter

- **Lowering `conf_threshold` doesn't clean this up.** There's no single
  weak box to rescue -- there are up to 39 competing, overlapping fragments
  per frame throughout the window. Dropping the threshold hands BoT-SORT a
  firehose of noisy candidates to disambiguate every single frame, which
  would plausibly make continuity worse, not better -- and since this is a
  global parameter, it would add that same noise to every other video in
  the dataset, not just this window.
- **Raising `max_gap_seconds` doesn't recover the missing footage.** It
  only controls whether the tracker stitches two EXISTING detections
  across a gap into one segment ID -- it cannot manufacture a trajectory
  entry for a frame that never had a usable detection. With no frame in
  this window producing a single confident box, there's nothing to
  interpolate.

## Conclusion

This looks like a genuine structural limitation of frame-based person
detection under motion blur, not a bug: **the exact moments a surveillance
system most wants to catch -- sudden, violent, fast motion -- are the
moments a standard object detector is least able to produce a single
confident, trackable box.** The faster and more violent the action, the
more it degrades the detector's ability to see it as a person at all,
which is a difficult, self-defeating property for a threat-detection
pipeline to have. Documented here rather than patched with a threshold
tweak, since the evidence above rules out the parameters that would
normally be reached for first.


In [ ]:
"""
Phase 2 -- diagnostic: visualize the exact crop VLM_USE_CROPS sends to the
VLM for one step of one segment, against the full uncropped frame, to
check by eye whether a relevant entity (an animal, a low-lying object,
anything outside the tracked person's own bounding box) falls outside the
cropped region -- and is therefore never shown to the VLM at all.

Motivated by Abuse030_x264.mp4: every generated summary described two
PEOPLE interacting, while every UCA reference caption for that video
describes a person and a dog/puppy -- no second person at all. Two
possible causes: (1) detect_people only ever detects/tracks COCO class
"person", so a dog is architecturally invisible as a tracked entity, or
(2) VLM_USE_CROPS crops tightly around the tracked PERSON's padded box,
which could exclude a dog positioned near the ground beside/behind them.
This diagnostic makes (2) checkable directly.

Cheap: only re-runs frame extraction + YOLO detection + segmentation (NO
VLM/Ollama calls) to reconstruct the per-frame bbox data that
phase2_output_*.json doesn't persist (run_phase2's saved output has
segment-level summaries, not the raw per-step frame_path/bbox trajectory).

Depends on: extract_frames/detect_people/process_segments (VLM-pipeline
prerequisites), the shared pipeline-driver cell (WORK_DIR/FRAMES_DIR/
MIN_SEGMENT_DURATION/clear_pipeline_dirs/_copy_video_locally), the
VLM-pipeline cell (_compute_crop_box/VLM_CROP_PAD_RATIO/
VLM_CROP_MIN_FRACTION), and the UCF-Crime shared-utilities cell
(get_video_fps) -- that last one has no UCF-specific logic in it, so this
works for a VIRAT video too if you pass VIRAT's VIDEO_DIR instead.
"""

import cv2


def visualize_crop_for_segment(video_file_name: str, video_dir: str,
                               segment_id: int = 1, step_index: int = 0,
                               out_dir: str = "/content/crop_diagnostics") -> None:
    """Re-run detection/segmentation for one video (no VLM calls) and save
    two images for one trajectory step of one segment:
      - the full frame, with the raw detection box (green) and the padded
        crop region actually sent to the VLM (red) drawn on it
      - the crop itself, exactly as _load_vlm_image_bytes would produce it

    so you can check by eye whether something relevant falls outside the
    red box -- and is therefore invisible to the VLM for that frame.
    """
    os.makedirs(out_dir, exist_ok=True)
    video_basename = os.path.splitext(video_file_name)[0]
    drive_video_path = os.path.join(video_dir, video_file_name)

    clear_pipeline_dirs()
    local_video_path = _copy_video_locally(drive_video_path, WORK_DIR)
    fps = get_video_fps(local_video_path)
    extract_frames(local_video_path, FRAMES_DIR, fps=round(fps, 3))
    try:
        os.remove(local_video_path)
    except OSError:
        pass

    detections = detect_people(FRAMES_DIR, fps=round(fps, 3))
    segments = process_segments(detections, min_duration_seconds=MIN_SEGMENT_DURATION)

    matches = [s for s in segments if s["segment_id"] == segment_id]
    if not matches:
        print(f"[visualize_crop_for_segment] no segment_id={segment_id} found "
              f"({len(segments)} segment(s) total for this video)")
        return
    segment = matches[0]

    if step_index >= len(segment["trajectory"]):
        print(f"[visualize_crop_for_segment] segment {segment_id} only has "
              f"{len(segment['trajectory'])} trajectory step(s), "
              f"step_index={step_index} is out of range")
        return
    step = segment["trajectory"][step_index]
    frame_path, bbox = step["frame_path"], step["bbox"]

    img = cv2.imread(frame_path)
    if img is None:
        print(f"[visualize_crop_for_segment] could not read {frame_path}")
        return
    h, w = img.shape[:2]

    x1, y1, x2, y2 = _compute_crop_box(bbox, w, h)

    overlay = img.copy()
    bx1, by1, bx2, by2 = bbox
    cv2.rectangle(overlay, (bx1, by1), (bx2, by2), (0, 255, 0), 2)   # green: raw detection box
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 255), 2)       # red: padded crop sent to the VLM

    crop = img[y1:y2, x1:x2]

    full_path = os.path.join(
        out_dir, f"{video_basename}_seg{segment_id}_step{step_index}_full_with_boxes.jpg")
    crop_path = os.path.join(
        out_dir, f"{video_basename}_seg{segment_id}_step{step_index}_crop.jpg")
    cv2.imwrite(full_path, overlay)
    cv2.imwrite(crop_path, crop)

    frame_area = w * h
    crop_area = (x2 - x1) * (y2 - y1)
    print(f"[visualize_crop_for_segment] frame: {frame_path}")
    print(f"  timestamp: {step['timestamp']:.1f}s")
    print(f"  full frame size: {w}x{h}")
    print(f"  raw detection bbox (green): {bbox}")
    print(f"  padded crop region sent to VLM (red): [{x1}, {y1}, {x2}, {y2}] "
          f"({x2 - x1}x{y2 - y1}, {100 * crop_area / frame_area:.1f}% of frame area)")
    print(f"  saved full-frame overlay: {full_path}")
    print(f"  saved actual crop:        {crop_path}")


if __name__ == "__main__":
    visualize_crop_for_segment("Abuse030_x264.mp4", UCF_VIDEO_DIR, segment_id=1, step_index=0)


In [ ]:
"""
Phase 1 -- diagnostic: dump per-frame YOLO person-detection confidence
across a specific time window, to tell apart three different explanations
for a gap between tracked segments:

  (a) genuinely zero person-candidates found at all (occlusion, out of
      frame, or a pose too atypical for the detector to propose a box for
      in the first place) -- no confidence threshold would fix this.
  (b) a candidate box exists but scores below detect_people's conf_threshold
      (0.5) -- fixable by lowering that threshold.
  (c) a candidate scores ABOVE 0.5 in this window -- if so, that frame
      SHOULD already appear in detect_people's output, and a gap here would
      point at the TRACKER (BoT-SORT continuity/ID association) rather
      than the detector, a different bug entirely.

Motivated by Abuse030_x264.mp4: the reference-confirmed abuse happens at
42-45s, squarely inside a 7.2s gap (40.3-47.5s) between two tracked
segments -- meaning no VLM call, regardless of model choice, ever saw that
footage. This checks WHY the tracker lost the person there.

Runs YOLO directly with conf=0.001 (i.e. no practical floor) so even
very weak candidate boxes show up, unlike detect_people() which only ever
returns detections that already cleared 0.5.

Depends on: extract_frames (frame-extraction cell), _get_yolo_model /
PERSON_CLASS_ID / _frame_index_from_path (person-detection cell), the
shared pipeline-driver cell (WORK_DIR/FRAMES_DIR/clear_pipeline_dirs/
_copy_video_locally), and get_video_fps (UCF-Crime shared-utilities cell --
no UCF-specific logic in it, works for any video).
"""

import csv


def diagnose_detection_confidence(video_file_name: str, video_dir: str,
                                  start_time: float, end_time: float,
                                  model_name: str = "yolov8n.pt",
                                  out_csv: str | None = None) -> list[dict]:
    """Print (and optionally save) per-frame person-detection confidence
    for every frame whose timestamp falls in [start_time, end_time].

    Extracts frames fresh (cheap, no VLM calls) rather than assuming
    FRAMES_DIR already holds the right video's frames.
    """
    video_basename = os.path.splitext(video_file_name)[0]
    drive_video_path = os.path.join(video_dir, video_file_name)

    clear_pipeline_dirs()
    local_video_path = _copy_video_locally(drive_video_path, WORK_DIR)
    fps = get_video_fps(local_video_path)
    extract_frames(local_video_path, FRAMES_DIR, fps=round(fps, 3))
    try:
        os.remove(local_video_path)
    except OSError:
        pass

    model = _get_yolo_model(model_name)
    frame_files = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg"))

    records = []
    for fname in frame_files:
        frame_path = os.path.join(FRAMES_DIR, fname)
        frame_idx = _frame_index_from_path(frame_path)
        timestamp = frame_idx / fps
        if not (start_time <= timestamp <= end_time):
            continue

        # conf=0.001, not 0 -- some ultralytics versions reject an exact 0
        # confidence floor. classes=[PERSON_CLASS_ID] keeps this comparable
        # to detect_people(), which is person-only.
        pred = model(frame_path, conf=0.001, classes=[PERSON_CLASS_ID], verbose=False)[0]
        boxes = pred.boxes
        if boxes is None or len(boxes) == 0:
            records.append({"frame_idx": frame_idx, "timestamp": round(timestamp, 3),
                            "n_candidates": 0, "max_confidence": None})
            continue

        confs = boxes.conf.cpu().numpy()
        records.append({
            "frame_idx": frame_idx,
            "timestamp": round(timestamp, 3),
            "n_candidates": int(len(confs)),
            "max_confidence": round(float(confs.max()), 4),
        })

    print(f"[diagnose_detection_confidence] {video_file_name}: {len(records)} frame(s) "
          f"checked in [{start_time}s, {end_time}s] (fps={fps:.3f})")
    print(f"{'frame':>8} {'time(s)':>9} {'n_cand':>7} {'max_conf':>9}")
    for r in records:
        conf_str = f"{r['max_confidence']:.4f}" if r["max_confidence"] is not None else "  none"
        flag = ""
        if r["max_confidence"] is not None and r["max_confidence"] < 0.5:
            flag = "  <-- below detect_people's 0.5 threshold"
        print(f"{r['frame_idx']:>8} {r['timestamp']:>9.3f} {r['n_candidates']:>7} {conf_str:>9}{flag}")

    n_zero = sum(1 for r in records if r["n_candidates"] == 0)
    n_below = sum(1 for r in records
                 if r["max_confidence"] is not None and r["max_confidence"] < 0.5)
    n_above = sum(1 for r in records
                 if r["max_confidence"] is not None and r["max_confidence"] >= 0.5)
    print(
        f"\n[diagnose_detection_confidence] summary: {n_zero} frame(s) with ZERO person "
        f"candidates at all (not fixable by conf_threshold), {n_below} with a candidate "
        f"below the 0.5 threshold (fixable by lowering conf_threshold), {n_above} already "
        f"above 0.5 (these frames should already be in detect_people's output -- if a gap "
        f"still exists here, that points at the TRACKER's continuity logic, not detection)"
    )

    if out_csv:
        os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
        with open(out_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["frame_idx", "timestamp", "n_candidates", "max_confidence"])
            writer.writeheader()
            writer.writerows(records)
        print(f"[diagnose_detection_confidence] wrote {out_csv}")

    return records


if __name__ == "__main__":
    diagnose_detection_confidence(
        "Abuse030_x264.mp4", UCF_VIDEO_DIR,
        start_time=39.0, end_time=48.0,
    )


In [ ]:
"""
Phase 2 -- UCF-Crime error analysis: per-category breakdown.

Reads the per-video rows already written to UCF_CSV_PATH (one row per
video, written by eval_video_ucf_crime during the full-dataset run) and
groups them by UCF-Crime category, since the dataset-wide summary
(precision/recall/F1/mean scores pooled across all 53 videos) can't show
WHICH crime categories the pipeline actually handles well versus falls
apart on -- e.g. whether "Arrest"-type videos are systematically worse
than "Fighting"-type ones, the way the case-by-case diagnosis in this
notebook suggested.

Depends on: the UCF-Crime shared-utilities cell (UCF_CSV_PATH). Reads an
existing CSV rather than running anything -- run this after
run_ucf_crime_pipeline has produced results.
"""

import pandas as pd


def summarize_by_category(csv_path: str | None = None) -> pd.DataFrame:
    """Group UCF_CSV_PATH's per-video rows by category and return a
    summary table: video count, suspicious-classification accuracy, mean
    risk score, and mean description-quality scores per category.

    Sorted worst-accuracy-first so the categories most worth investigating
    are at the top rather than buried alphabetically.
    """
    csv_path = csv_path or UCF_CSV_PATH
    df = pd.read_csv(csv_path)

    # CSV round-trips booleans as the literal strings "True"/"False" --
    # coerce explicitly rather than trusting pandas' type inference, which
    # can vary by version / column contents rather than reliably giving a
    # real bool dtype.
    for col in ("gt_suspicious", "pred_suspicious", "correct_suspicious_classification"):
        if df[col].dtype == object:
            df[col] = df[col].map({"True": True, "False": False}).astype(bool)

    # If a video was processed more than once (e.g. spot_check_videos
    # re-running it after a prompt change), keep only its LAST row so a
    # re-run doesn't double-count that video in the per-category stats.
    df = df.drop_duplicates(subset="video", keep="last")

    summary = df.groupby("category").agg(
        num_videos=("video", "count"),
        num_correct=("correct_suspicious_classification", "sum"),
        accuracy=("correct_suspicious_classification", "mean"),
        mean_video_risk_score=("video_risk_score", "mean"),
        mean_summary_faithfulness=("mean_summary_faithfulness", "mean"),
        mean_summary_agreement=("mean_summary_agreement", "mean"),
        mean_num_segments=("num_segments", "mean"),
        mean_num_segments_with_reference=("num_segments_with_reference", "mean"),
    ).round(3)

    summary = summary.sort_values("accuracy")
    return summary


if __name__ == "__main__":
    table = summarize_by_category()
    print(table.to_string())
